# บทที่ 2 — ระเบียบวิธีประเมินผลระบบผู้แนะนำ
## Lab 2 · ฉบับรวมไฟล์เดียว (All-in-One) สำหรับ Google Colab

> **ประโยคสำคัญของทั้งบท**
> คะแนนประเมินไม่ได้เป็นคุณสมบัติของแบบจำลองเพียงอย่างเดียว แต่เป็นผลร่วมของ
> **แบบจำลอง · ข้อมูล · ตัวชี้วัด · เวลา · เซตผู้สมัคร · และโปรโตคอลการประเมินทั้งหมด**

---

### สมุดนี้คืออะไร

รวมทุกไฟล์ของโฟลเดอร์ `Lab2` เดิม (แพ็กเกจ `rslab` 6 โมดูล + `make_dataset.py` + สคริปต์การทดลอง `lab2_01` ถึง `lab2_07`)
ไว้ใน **ไฟล์เดียว** เปิดบน Colab แล้วกด `Runtime → Run all` ได้ทันที
**ไม่ต้องอัปโหลดโฟลเดอร์ ไม่ต้อง mount Google Drive ไม่ต้องติดตั้งไลบรารีเพิ่ม**

### แผนผังของสมุด

| ส่วน | หัวข้อ | ตรงกับไฟล์เดิม | ตอบคำถามว่า |
|---|---|---|---|
| 0 | เตรียมสภาพแวดล้อม + ไลบรารี `rslab` | `rslab/*.py` | — |
| 0.5 | สร้างชุดข้อมูล | `make_dataset.py` | — |
| 1 | MAE และ RMSE | `lab2_01` | ทายคะแนนพลาดกี่หน่วย |
| 2 | ตัวชี้วัดเชิงอันดับแปดตัว | `lab2_02` | ตัวชี้วัดใดเห็น "ตำแหน่ง" |
| 3 | การแบ่งข้อมูลตามเวลาและการรั่วไหล | `lab2_03` | แบ่งผิดทำให้เกินจริงเท่าใด |
| 4 | ฟูลแค็ตตาล็อก เทียบ การสุ่มรายการเชิงลบ ⭐ | `lab2_04` | การสุ่มเปลี่ยน "ผู้ชนะ" ได้หรือไม่ |
| 5 | คุณภาพนอกเหนือความถูกต้อง | `lab2_05` | แม่นที่สุด = ดีที่สุด จริงหรือ |
| 6 | ความไม่แน่นอนและนัยสำคัญ | `lab2_06` | ส่วนต่างที่เห็นเป็นของจริงหรือสัญญาณรบกวน |
| 7 | โปรโตคอลการประเมิน 11 ขั้น | `lab2_07` | รายงานอย่างซื่อสัตย์ทำอย่างไร |
| **8** | **เปรียบเทียบ Performance ระหว่างแบบจำลอง 5 ตัว** ⭐ | **(ใหม่)** | **Random · TopPopular · ItemKNN · UserKNN · MF-SVD** |
| 9 | สรุปและบัตรโปรโตคอล | — | — |

### ไฟล์ประกอบชุดนี้

* `Lab2_All_in_One.ipynb` — สมุดเล่มนี้ (เนื้อหา + การทดลอง + เฉลยตัวอย่าง)
* `Lab2_Exercises.ipynb` — **แบบฝึกหัด 12 ข้อ** ให้ทดลองเขียนเอง มีเซลล์ตรวจคำตอบอัตโนมัติ
* `Lab2_Exercises_Solutions.ipynb` — **เฉลย** พร้อมคำอธิบายทีละข้อ

> เวลารันทั้งเล่มโดยประมาณ 2–4 นาทีบน Colab ฟรี (ส่วนที่ช้าที่สุดคือการจำลอง Monte Carlo ในส่วนที่ 4)

---
## ส่วนที่ 0 — เตรียมสภาพแวดล้อมและไลบรารี `rslab`

สามเซลล์ถัดไปคือเนื้อหาของแพ็กเกจ `rslab` ทั้งหมด ย้ายมาไว้ในสมุดโดยตรง

| เซลล์ | เทียบเท่าไฟล์เดิม | มีอะไร |
|---|---|---|
| 0.1 | — | นำเข้าไลบรารีและตั้งค่ากราฟ |
| 0.2 | `rslab/metrics_rating.py` + `rslab/metrics_ranking.py` | MAE, RMSE, P@K, R@K, F1, HR, RR/MRR, AP/MAP, DCG/nDCG, AUC |
| 0.3 | `rslab/metrics_beyond.py` | Coverage, Gini, ILD, Novelty, LongTail, Calibration, Fairness |
| 0.4 | `rslab/splitters.py` + `significance.py` + `protocol.py` | การแบ่งข้อมูล 4 แบบ, paired bootstrap, Holm, ProtocolCard |

> **หมายเหตุเรื่องภาษาในกราฟ** ข้อความอธิบายทั้งหมดเป็นภาษาไทย แต่ *ป้ายกำกับในกราฟ* ใช้ภาษาอังกฤษโดยเจตนา
> เพราะ Colab ไม่ได้ติดตั้งฟอนต์ไทยให้ matplotlib มาตั้งแต่ต้น การใช้อังกฤษจึงทำให้กราฟไม่กลายเป็นสี่เหลี่ยมว่าง

In [ ]:
# ============================================================================
# เซลล์ 0.1  นำเข้าไลบรารีและตั้งค่าส่วนกลาง
# ไม่ต้อง pip install อะไรเพิ่ม Colab มี numpy / pandas / matplotlib / scipy ครบแล้ว
# ============================================================================
from __future__ import annotations

import math
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ตั้งค่าการแสดงผลตาราง
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# ตั้งค่ากราฟ: ป้ายกำกับในกราฟใช้ภาษาอังกฤษ จึงไม่ต้องติดตั้งฟอนต์ไทย
plt.rcParams.update({
    "figure.figsize": (9, 4.2),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

SEED = 2569
rng_global = np.random.default_rng(SEED)

print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("เมล็ดสุ่มหลักของสมุดนี้ :", SEED)
print("\nพร้อมใช้งาน — รันเซลล์ถัดไปได้เลย")

In [ ]:
# ============================================================================
# เซลล์ 0.2  ตัวชี้วัดการพยากรณ์คะแนน และตัวชี้วัดเชิงอันดับ
# รวมเนื้อหาของ rslab/metrics_rating.py + rslab/metrics_ranking.py
#
# ข้อตกลงร่วมของทุกฟังก์ชันเชิงอันดับ
#   ranked   : ลำดับรายการที่ระบบแนะนำ เรียงจากอันดับ 1 ไปท้าย เช่น ["i9","i12","i8"]
#   relevant : เซตรายการที่ผู้ใช้ชอบจริงในชุดทดสอบ            เช่น {"i7","i9"}
#   K        : ค่าตัดอันดับ ต้องประกาศทุกครั้งที่รายงานผล
# ============================================================================

# ------------------------------------------------------- 1) Rating Prediction
def mae(y_true, y_pred) -> float:
    """MAE = (1/n) Σ |y_true − y_pred|   —  "โดยเฉลี่ยทายพลาดไปกี่ดาว"

    ให้น้ำหนักความผิดพลาดทุกขนาดเท่ากัน (เชิงเส้น) จึงทนต่อค่าผิดปกติได้ดีกว่า RMSE
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true และ y_pred ต้องมีขนาดเท่ากัน")
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred) -> float:
    """RMSE = sqrt( (1/n) Σ (y_true − y_pred)² )

    การยกกำลังสองทำให้ความผิดพลาด "ก้อนใหญ่" ถูกลงโทษหนักกว่าหลายก้อนเล็ก
    RMSE ≥ MAE เสมอ (อสมการของเยนเซน) และเท่ากันเมื่อความผิดพลาดทุกตัวขนาดเท่ากัน
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true และ y_pred ต้องมีขนาดเท่ากัน")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def error_table(y_true, y_pred, labels=None):
    """ตารางความคลาดเคลื่อนทีละแถว (label, y_true, y_pred, error, |error|, error²)"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    e = y_pred - y_true
    if labels is None:
        labels = [f"#{i+1}" for i in range(len(y_true))]
    return [{"label": lb, "y_true": t, "y_pred": p, "error": err,
             "abs_error": abs(err), "sq_error": err ** 2}
            for lb, t, p, err in zip(labels, y_true, y_pred, e)]


# ------------------------------------- 2) Set-based Metrics (ไม่ไวต่อตำแหน่ง)
def precision_at_k(ranked, relevant, k: int) -> float:
    """P@K = (จำนวนรายการเกี่ยวข้องใน Top-K) / K   — "ของที่เสิร์ฟไป มีกี่ชิ้นที่ใช่"

    ตัวส่วนเป็น K คงที่เสมอ ดังนั้น P@K จึงมีแนวโน้ม *ลดลง* เมื่อ K เพิ่ม
    """
    top = list(ranked)[:k]
    return sum(1 for it in top if it in relevant) / k


def recall_at_k(ranked, relevant, k: int) -> float:
    """R@K = (จำนวนรายการเกี่ยวข้องใน Top-K) / |ℛᵤ|  — "ของที่ชอบจริง หยิบมาได้กี่ส่วน"

    ไม่มีวันลดลงเมื่อ K เพิ่ม (ตัวเศษเพิ่มได้ ตัวส่วนคงที่)
    """
    relevant = set(relevant)
    if not relevant:
        return float("nan")
    top = list(ranked)[:k]
    return sum(1 for it in top if it in relevant) / len(relevant)


def f1_at_k(ranked, relevant, k: int) -> float:
    """F1 = 2PR/(P+R)  ค่าเฉลี่ยฮาร์มอนิก จึงถูกดึงลงโดยด้านที่ต่ำกว่า"""
    p = precision_at_k(ranked, relevant, k)
    r = recall_at_k(ranked, relevant, k)
    if np.isnan(r) or (p + r) == 0:
        return 0.0
    return 2 * p * r / (p + r)


def hit_rate_at_k(ranked, relevant, k: int) -> float:
    """HR@K = 1 ถ้ามีรายการเกี่ยวข้องอย่างน้อยหนึ่งชิ้นใน Top-K มิฉะนั้น 0"""
    top = list(ranked)[:k]
    return 1.0 if any(it in relevant for it in top) else 0.0


# ------------------------------------ 3) Rank-aware Metrics (ไวต่อตำแหน่ง)
def reciprocal_rank(ranked, relevant, k: int | None = None) -> float:
    """RR = 1 / (อันดับของรายการเกี่ยวข้อง *ชิ้นแรก*)  ถ้าไม่พบเลย = 0

    เฉลี่ยข้ามผู้ใช้แล้วได้ MRR  เหมาะกับงานที่ต้องการคำตอบดีเพียงรายการแรก
    จุดอ่อน: หลังพบชิ้นแรกแล้ว คะแนนไม่เปลี่ยนอีกเลย
    """
    seq = list(ranked) if k is None else list(ranked)[:k]
    for pos, it in enumerate(seq, start=1):
        if it in relevant:
            return 1.0 / pos
    return 0.0


def average_precision_at_k(ranked, relevant, k: int) -> float:
    """AP@K = ( Σ_{p: rel(p)=1} Precision@p ) / min(|ℛᵤ|, K)

    ต่างจาก RR ตรงที่ใช้รายการเกี่ยวข้อง "ทุกชิ้น" ใน Top-K
    ตัวส่วน min(K, |ℛᵤ|) กันการหารด้วยจำนวนที่เป็นไปไม่ได้
    """
    relevant = set(relevant)
    if not relevant:
        return float("nan")
    seq = list(ranked)[:k]
    hits, score = 0, 0.0
    for pos, it in enumerate(seq, start=1):
        if it in relevant:
            hits += 1
            score += hits / pos            # = Precision@pos ณ จุดที่เจอ
    return score / min(len(relevant), k)


def dcg_at_k(gains, k: int, gain_type: str = "linear") -> float:
    """DCG@K = Σ_{p=1..K} g(rel_p) / log2(p+1)

    gain_type = "linear"      -> g(rel) = rel        (นิยมกับความเกี่ยวข้องทวิภาค)
    gain_type = "exponential" -> g(rel) = 2^rel − 1  (มาตรฐาน TREC)
    *ต้องประกาศ gain_type ทุกครั้ง* เพราะสองแบบให้ค่าต่างกันมากเมื่อ rel > 1
    """
    g = np.asarray(gains, dtype=float)[:k]
    if gain_type == "exponential":
        g = np.power(2.0, g) - 1.0
    elif gain_type != "linear":
        raise ValueError("gain_type ต้องเป็น 'linear' หรือ 'exponential'")
    discounts = 1.0 / np.log2(np.arange(2, len(g) + 2))
    return float(np.sum(g * discounts))


def ndcg_at_k(ranked, relevance: dict, k: int, gain_type: str = "linear") -> float:
    """nDCG@K = DCG@K / IDCG@K   ค่าอยู่ใน [0, 1] เสมอ

    relevance : dict {item: rel} รายการที่ไม่อยู่ใน dict ถือว่า rel = 0
    IDCG มาจากการเรียง rel จากมากไปน้อย จึงขึ้นกับ "ผู้ใช้" ไม่ใช่ "แบบจำลอง"
    """
    gains = [relevance.get(it, 0.0) for it in list(ranked)[:k]]
    dcg = dcg_at_k(gains, k, gain_type)
    idcg = dcg_at_k(sorted(relevance.values(), reverse=True), k, gain_type)
    return 0.0 if idcg == 0 else dcg / idcg


# --------------------------------------------------------------- 4) AUC
def auc_from_ranking(ranked, relevant) -> float:
    """AUC = (จำนวนคู่ (เกี่ยวข้อง, ไม่เกี่ยวข้อง) ที่เรียงถูก) / (จำนวนคู่ทั้งหมด)

    ตีความ: ความน่าจะเป็นที่รายการเกี่ยวข้องซึ่งสุ่มมา จะถูกจัดสูงกว่ารายการไม่เกี่ยวข้องซึ่งสุ่มมา
    0.5 = เดาสุ่ม   ข้อจำกัด: ให้น้ำหนักทุกคู่เท่ากัน จึงไม่เหมาะเป็น Primary Metric ของงาน Top-K
    """
    ranked = list(ranked)
    rel_pos = [i for i, it in enumerate(ranked) if it in relevant]
    irr_pos = [i for i, it in enumerate(ranked) if it not in relevant]
    if not rel_pos or not irr_pos:
        return float("nan")
    good = sum(1 for rp in rel_pos for ip in irr_pos if rp < ip)
    return good / (len(rel_pos) * len(irr_pos))


def roc_curve_from_ranking(ranked, relevant):
    """คืน (fpr, tpr) เพื่อวาดเส้นโค้ง ROC จากการจัดอันดับหนึ่งชุด"""
    ranked = list(ranked)
    P = sum(1 for it in ranked if it in relevant)
    N = len(ranked) - P
    tp = fp = 0
    fpr, tpr = [0.0], [0.0]
    for it in ranked:
        if it in relevant:
            tp += 1
        else:
            fp += 1
        tpr.append(tp / P if P else 0.0)
        fpr.append(fp / N if N else 0.0)
    return fpr, tpr


def evaluate_user(ranked, relevance: dict, k: int, gain_type: str = "linear") -> dict:
    """คำนวณตัวชี้วัดเชิงอันดับทั้งแปดตัวของผู้ใช้หนึ่งคนในครั้งเดียว"""
    relevant = {it for it, r in relevance.items() if r > 0}
    return {
        "P@K":    precision_at_k(ranked, relevant, k),
        "R@K":    recall_at_k(ranked, relevant, k),
        "F1@K":   f1_at_k(ranked, relevant, k),
        "HR@K":   hit_rate_at_k(ranked, relevant, k),
        "RR":     reciprocal_rank(ranked, relevant, k),
        "AP@K":   average_precision_at_k(ranked, relevant, k),
        "nDCG@K": ndcg_at_k(ranked, relevance, k, gain_type),
        "AUC":    auc_from_ranking(ranked, relevant),
    }


print("✓ เซลล์ 0.2 พร้อม : mae, rmse, precision_at_k, recall_at_k, f1_at_k, hit_rate_at_k,")
print("                  reciprocal_rank, average_precision_at_k, dcg_at_k, ndcg_at_k,")
print("                  auc_from_ranking, roc_curve_from_ranking, evaluate_user")

In [ ]:
# ============================================================================
# เซลล์ 0.3  ตัวชี้วัดนอกเหนือความถูกต้อง (Beyond-Accuracy)
# รวมเนื้อหาของ rslab/metrics_beyond.py
#
# ระบบที่แม่นยำที่สุดอาจเป็นระบบที่แนะนำของยอดนิยมซ้ำ ๆ ให้ทุกคน
# ตัวชี้วัดกลุ่มนี้จึงวัดสิ่งที่ความแม่นยำมองไม่เห็น
# ============================================================================

# ------------------------------------------------------------ ความครอบคลุม
def catalog_coverage(all_reclists, n_items: int) -> float:
    """Coverage@K = |{รายการที่ปรากฏใน Top-K ของใครสักคน}| / |ℐ|

    ยูเนียนของเซต -> รายการหนึ่งชิ้นนับครั้งเดียว ไม่ว่าจะโผล่ในผลของกี่คน
    นี่คือทั้งจุดแข็งและจุดอ่อน: บอกได้ว่า "กี่ชิ้นเคยถูกเห็น" แต่บอกไม่ได้ว่า "กระจายแค่ไหน"
    """
    shown = set()
    for lst in all_reclists:
        shown.update(lst)
    return len(shown) / n_items


def gini_index(all_reclists, n_items: int) -> float:
    """สัมประสิทธิ์จีนีของการเปิดเผย  0 = เท่าเทียม, ใกล้ 1 = กระจุกตัวสุดขั้ว

        G = Σ (2i − n − 1)·x_i  /  ( n · Σ x_i )      โดย x เรียงจากน้อยไปมาก

    กฎบังคับสองข้อ
      1. ต้องเรียง x จากน้อยไปมากก่อนเสมอ ลืมเรียงแล้วสูตรคืนค่าที่ไม่มีความหมาย
      2. รายการที่ "ไม่เคยถูกแนะนำเลย" ต้องอยู่ในเวกเตอร์ในฐานะ 0
         ถ้าตัดทิ้ง ค่าจีนีจะต่ำกว่าความจริง เพราะกลุ่มที่ถูกกีดกันที่สุดหายไปพอดี
    ตรวจทานง่าย ๆ : Σ(2i − n − 1) = 0 เสมอ
    """
    cnt = Counter()
    for lst in all_reclists:
        cnt.update(lst)
    vals = np.array(sorted(list(cnt.values()) + [0] * (n_items - len(cnt))), dtype=float)
    n = len(vals)
    if vals.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * vals).sum() / (n * vals.sum()))


def gini_trace(values):
    """คืน (x เรียงแล้ว, สัมประสิทธิ์ 2i−n−1, ผลคูณ, ค่าจีนี) ไว้พิมพ์ตรวจทีละพจน์ด้วยมือ"""
    v = np.array(sorted(values), dtype=float)
    n = len(v)
    i = np.arange(1, n + 1)
    w = 2 * i - n - 1
    g = float((v * w).sum() / (n * v.sum())) if v.sum() else 0.0
    return v, w, v * w, g


def lorenz_points(values):
    """จุดของเส้นโค้งลอเรนซ์ (สัดส่วนรายการสะสม, สัดส่วนการเปิดเผยสะสม)"""
    v = np.array(sorted(values), dtype=float)
    if v.sum() == 0:
        return np.linspace(0, 1, len(v) + 1), np.linspace(0, 1, len(v) + 1)
    x = np.arange(0, len(v) + 1) / len(v)
    y = np.concatenate([[0.0], np.cumsum(v) / v.sum()])
    return x, y


# ------------------------------------------------------------ ความหลากหลาย
def intra_list_diversity(reclist, sim_fn) -> float:
    """ILD = ค่าเฉลี่ยของ (1 − sim(i, j)) ข้ามทุกคู่ในรายการเดียวกัน

    ตัวประกอบ 2/(K(K−1)) คือหนึ่งส่วนจำนวนคู่ทั้งหมด
    ILD วัดความหลากหลาย "ภายในรายการของผู้ใช้คนเดียว" — ไม่เคยเห็นผู้ใช้คนอื่นเลย
    จึงต้องรายงานคู่กับ Coverage และ Gini เสมอ
    """
    items = list(reclist)
    if len(items) < 2:
        return 0.0
    tot, cnt = 0.0, 0
    for a in range(len(items)):
        for b in range(a + 1, len(items)):
            tot += 1.0 - sim_fn(items[a], items[b])
            cnt += 1
    return tot / cnt


def category_sim(cat_of: dict):
    """sim_fn อย่างง่าย: 1 ถ้าอยู่หมวด/แนวเดียวกัน มิฉะนั้น 0"""
    def _sim(i, j):
        return 1.0 if cat_of.get(i) == cat_of.get(j) else 0.0
    return _sim


# ------------------------------------------------------ ความใหม่ / หางยาว
def novelty(reclist, popularity: dict, n_users: int) -> float:
    """Novelty = (1/K) Σ −log₂ p(i)   โดย p(i) = ผู้เคยมีปฏิสัมพันธ์กับ i / |U|

    −log₂ p(i) คือ Self-Information หน่วยเป็น "บิต"
    เปลี่ยนไปใช้ ln ค่าจะเปลี่ยนทันที จึง *ต้องประกาศฐานลอการิทึมเสมอ*
    ช่องโหว่: แทนรายการเดียวด้วยของที่แทบไม่มีใครดู ก็ดันค่าเฉลี่ยขึ้นได้มาก
    """
    vals = []
    for it in reclist:
        p = popularity.get(it, 0) / n_users
        p = max(p, 1.0 / (n_users * 10))          # กันกรณี log(0)
        vals.append(-np.log2(p))
    return float(np.mean(vals)) if vals else 0.0


def long_tail_ratio(reclist, tail_items: set) -> float:
    """สัดส่วนของรายการส่วนหางใน Top-K"""
    lst = list(reclist)
    if not lst:
        return 0.0
    return sum(1 for it in lst if it in tail_items) / len(lst)


def head_tail_split(popularity: dict, head_share: float = 0.33):
    """แบ่งคลังเป็นกลุ่มหัว/หาง ตามสัดส่วนปฏิสัมพันธ์สะสม (เกณฑ์หนึ่งในสาม)

    "หัว" และ "หาง" ไม่มีเกณฑ์ตายตัว (Top-2, 20% ของรายการ, หรือ 80% ของการรับชม)
    จึงต้องประกาศวิธีแบ่งทุกครั้ง คืน (head_items, tail_items)
    """
    items = sorted(popularity.items(), key=lambda kv: -kv[1])
    total = sum(popularity.values())
    head, acc = set(), 0
    for it, c in items:
        if acc >= head_share * total:
            break
        head.add(it)
        acc += c
    return head, set(popularity) - head


# ------------------------------------------------------------- การปรับเทียบ
def calibration_error(reclist, user_profile, cat_of: dict) -> float:
    """CalibrationError = 0.5 · Σ_c |q(c) − p(c)|   (ระยะทางรวมแปรผัน, Steck 2018)

    p(c) = สัดส่วนหมวด c ในประวัติผู้ใช้   q(c) = สัดส่วนหมวด c ในรายการที่แนะนำ
    ค่าใกล้ 0 = คำแนะนำสอดคล้องกับประวัติ   ค่า 0.20 กับ K=10 ≈ ต้องปรับ 2 จาก 10 รายการ
    """
    def dist(items):
        c = Counter(cat_of.get(i, "?") for i in items)
        tot = sum(c.values())
        return {k: v / tot for k, v in c.items()} if tot else {}
    p = dist(list(user_profile))
    q = dist(list(reclist))
    return 0.5 * sum(abs(q.get(c, 0.0) - p.get(c, 0.0)) for c in set(p) | set(q))


# ------------------------------------------------------------- ความเป็นธรรม
def group_parity_gap(scores_by_group: dict) -> float:
    """User-Side Fairness: gap = max_g mean(score_g) − min_g mean(score_g)

    กับดัก Leveling Down: ระบบที่แย่เท่ากันกับทุกคนได้ gap = 0 "สมบูรณ์แบบ"
    จึงต้องรายงานค่าเฉลี่ยรายกลุ่มควบคู่กับช่องว่างเสมอ
    """
    means = {g: float(np.mean(v)) for g, v in scores_by_group.items() if len(v)}
    if len(means) < 2:
        return 0.0
    return max(means.values()) - min(means.values())


def provider_exposure_gap(all_reclists, provider_of: dict) -> float:
    """Provider-Side Fairness: ค่าจีนีของการเปิดเผย *รายผู้ผลิต*

    เวกเตอร์ต้องตั้งต้นจากผู้ผลิตทุกรายใน provider_of ไม่ใช่จากที่ปรากฏในผลลัพธ์
    มิฉะนั้นค่ายที่ไม่เคยถูกแนะนำเลยจะหายไปจากการคำนวณ และค่าจีนีจะต่ำกว่าจริง
    """
    cnt = Counter({p: 0 for p in set(provider_of.values())})
    for lst in all_reclists:
        for it in lst:
            cnt[provider_of.get(it, "?")] += 1
    vals = np.array(sorted(cnt.values()), dtype=float)
    n = len(vals)
    if n < 2 or vals.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * vals).sum() / (n * vals.sum()))


print("✓ เซลล์ 0.3 พร้อม : catalog_coverage, gini_index, gini_trace, lorenz_points,")
print("                  intra_list_diversity, category_sim, novelty, long_tail_ratio,")
print("                  head_tail_split, calibration_error, group_parity_gap,")
print("                  provider_exposure_gap")

In [ ]:
# ============================================================================
# เซลล์ 0.4  การแบ่งข้อมูล · การทดสอบนัยสำคัญ · บัตรโปรโตคอล
# รวมเนื้อหาของ rslab/splitters.py + rslab/significance.py + rslab/protocol.py
#
# หลักการเดียวที่ห้ามละเมิด:
#   เหตุการณ์ในชุดฝึกต้องเกิด "ก่อน" เหตุการณ์ในชุดทดสอบ
# ============================================================================

# ------------------------------------------------- 1) การแบ่งข้อมูลสี่แบบ
def random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42):
    """แบ่งแบบสุ่ม — **ใช้เป็นตัวอย่างของสิ่งที่ไม่ควรทำเท่านั้น**

    ทำให้ชุดฝึกมีเหตุการณ์ที่เกิดหลังเหตุการณ์ในชุดทดสอบ = ใช้อนาคตทำนายอดีต
    """
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    n_test = int(len(df) * test_frac)
    mask = np.zeros(len(df), dtype=bool)
    mask[idx[:n_test]] = True
    return df[~mask].copy(), df[mask].copy()


def global_temporal_split(df: pd.DataFrame, train_frac: float = 0.8, valid_frac: float = 0.0):
    """จุดตัดเวลาจุดเดียวทั้งระบบ ก่อน = ฝึก หลัง = ทดสอบ

    สะท้อนการใช้งานจริงมากที่สุด คืน (train, valid, test)
    """
    d = df.sort_values("timestamp").reset_index(drop=True)
    n = len(d)
    i1 = int(n * train_frac)
    i2 = int(n * (train_frac + valid_frac))
    return d.iloc[:i1].copy(), d.iloc[i1:i2].copy(), d.iloc[i2:].copy()


def per_user_temporal_split(df: pd.DataFrame, test_frac: float = 0.2):
    """กันส่วนท้าย test_frac ของแต่ละคนไว้ทดสอบ

    ข้อดี: ทุกคนมีข้อมูลทดสอบแน่นอน   ข้อเสีย: จุดตัดไม่ตรงกัน จึงยังรั่ว "ข้ามผู้ใช้"
    """
    tr, te = [], []
    for _, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")
        n_test = max(1, int(round(len(g) * test_frac)))
        if len(g) <= n_test:
            tr.append(g)
            continue
        tr.append(g.iloc[:-n_test])
        te.append(g.iloc[-n_test:])
    train = pd.concat(tr).reset_index(drop=True) if tr else df.iloc[0:0]
    test = pd.concat(te).reset_index(drop=True) if te else df.iloc[0:0]
    return train, test


def leave_one_out_split(df: pd.DataFrame):
    """กันเหตุการณ์ล่าสุดของแต่ละคนไว้หนึ่งชิ้น

    นิยมในงานตีพิมพ์ แต่ความแปรปรวนสูงมาก เพราะตัดสินผู้ใช้ด้วยเหตุการณ์เดียว
    """
    tr, te = [], []
    for _, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")
        if len(g) < 2:
            tr.append(g)
            continue
        tr.append(g.iloc[:-1])
        te.append(g.iloc[-1:])
    train = pd.concat(tr).reset_index(drop=True) if tr else df.iloc[0:0]
    test = pd.concat(te).reset_index(drop=True) if te else df.iloc[0:0]
    return train, test


def leakage_report(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """นับเหตุการณ์ในชุดฝึกที่เกิด *หลัง* เหตุการณ์ที่เก่าที่สุดในชุดทดสอบ"""
    if len(train) == 0 or len(test) == 0:
        return {"verdict": "ข้อมูลไม่พอตรวจ"}
    t0 = test["timestamp"].min()
    n_future = int((train["timestamp"] > t0).sum())
    return {
        "n_train": len(train), "n_test": len(test), "test_starts_at": float(t0),
        "n_future_in_train": n_future, "pct_future": 100.0 * n_future / len(train),
        "train_time_range": (float(train["timestamp"].min()), float(train["timestamp"].max())),
        "test_time_range": (float(test["timestamp"].min()), float(test["timestamp"].max())),
        "verdict": "ปลอดภัย" if n_future == 0 else "พบการรั่วไหล",
    }


def k_core_filter(df: pd.DataFrame, k: int, verbose: bool = False):
    """กรองแบบ k-core วนซ้ำจนถึงจุดคงที่ พร้อมบันทึกจำนวนแถวทุกรอบ"""
    rounds, cur = [], df
    while True:
        uc = cur.groupby("user_id")["item_id"].transform("size")
        ic = cur.groupby("item_id")["user_id"].transform("size")
        keep = (uc >= k) & (ic >= k)
        rounds.append({"n_rows": len(cur), "n_users": cur.user_id.nunique(),
                       "n_items": cur.item_id.nunique(), "dropped": int((~keep).sum())})
        if keep.all() or cur.empty:
            break
        cur = cur[keep]
    if verbose:
        for i, r in enumerate(rounds):
            print(f"  รอบที่ {i}: แถว {r['n_rows']:,}  ผู้ใช้ {r['n_users']:,}  "
                  f"รายการ {r['n_items']:,}  ตัดออก {r['dropped']:,}")
    return cur, rounds


# --------------------------------------------- 2) ความไม่แน่นอนและนัยสำคัญ
def paired_bootstrap(scores_a, scores_b, n_boot: int = 2000, seed: int = 42,
                     alpha: float = 0.05) -> dict:
    """สุ่มซ้ำแบบจับคู่ *ระดับผู้ใช้*

    scores_a, scores_b ต้องเป็นคะแนนรายผู้ใช้ที่เรียงตรงกันคนต่อคน
    หน่วยของการสุ่มซ้ำต้องเป็นหน่วยที่อิสระต่อกัน ซึ่งในระบบผู้แนะนำคือ "ผู้ใช้"
    ไม่ใช่ "เหตุการณ์" มิฉะนั้นช่วงเชื่อมั่นจะแคบเกินจริงอย่างเป็นระบบ
    """
    a = np.asarray(scores_a, dtype=float)
    b = np.asarray(scores_b, dtype=float)
    if a.shape != b.shape:
        raise ValueError("จำนวนผู้ใช้ของสองแบบจำลองต้องเท่ากันและเรียงตรงกัน")
    d = a - b
    n = len(d)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boots = d[idx].mean(axis=1)
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"n_users": n, "mean_a": float(a.mean()), "mean_b": float(b.mean()),
            "mean_diff": float(d.mean()), "ci_low": lo, "ci_high": hi,
            "p_value": float(min(p, 1.0)), "significant": bool(lo > 0 or hi < 0),
            "pct_users_worse": float((d < 0).mean())}


def wilcoxon_signed_rank(scores_a, scores_b) -> dict:
    """การทดสอบวิลค็อกซันแบบจับคู่ (ไม่ต้องสมมติการแจกแจง) — ใช้ *ยืนยัน* ผล bootstrap"""
    try:
        from scipy.stats import wilcoxon
    except ImportError:
        return {"error": "ต้องติดตั้ง scipy ก่อน"}
    a = np.asarray(scores_a, dtype=float)
    b = np.asarray(scores_b, dtype=float)
    d = a - b
    if np.allclose(d, 0):
        return {"statistic": 0.0, "p_value": 1.0, "note": "ผลต่างเป็นศูนย์ทุกคน"}
    stat, p = wilcoxon(a, b, zero_method="wilcox")
    return {"statistic": float(stat), "p_value": float(p), "n_nonzero": int((d != 0).sum())}


def holm_correction(pvalues: dict, alpha: float = 0.05) -> dict:
    """Holm–Bonferroni: เรียง p จากน้อยไปมาก เทียบกับ α/(m − i)

    เมื่อคู่ใดไม่ผ่าน คู่ที่เหลือทั้งหมดถือว่าไม่มีนัยสำคัญ (ขั้นบันไดลง)
    ที่ α = 0.05 และ 10 คู่ โอกาสพบนัยสำคัญปลอมอย่างน้อยหนึ่งคู่สูงถึง 40.1%
    """
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m = len(items)
    out, reject_all = {}, True
    for i, (name, p) in enumerate(items):
        thr = alpha / (m - i)
        rej = reject_all and (p <= thr)
        if not rej:
            reject_all = False
        out[name] = {"p": p, "threshold": thr, "significant": rej}
    return out


def seed_variance(run_fn, seeds=(0, 1, 2, 3, 4)) -> dict:
    """รันซ้ำหลายเมล็ดสุ่มแล้วรายงานค่าเฉลี่ยและความผันผวน

    กฎที่ใช้ได้เสมอ: ถ้าส่วนต่างระหว่างสองการตั้งค่า *เล็กกว่า* SD ระหว่างเมล็ดสุ่ม
    ของการตั้งค่าเดียวกัน แสดงว่าไม่มีหลักฐานว่าต่างกัน
    """
    vals = np.array([run_fn(s) for s in seeds], dtype=float)
    sd = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
    ci = 1.96 * sd / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
    return {"values": vals.tolist(), "mean": float(vals.mean()), "sd": sd,
            "ci95_halfwidth": float(ci), "range": (float(vals.min()), float(vals.max()))}


# ------------------------------------------------------- 3) บัตรโปรโตคอล
@dataclass
class ProtocolCard:
    """บันทึกทุกการตัดสินใจที่ทำให้ตัวเลขเป็นตัวเลขนั้น

    ผู้อ่านที่มีเพียงบัตรใบนี้กับชุดข้อมูล ต้องทำซ้ำผลได้ครบทุกหลัก
    กรอกให้ครบ "ก่อน" รันครั้งแรก และแนบไปกับผลทุกครั้ง
    """
    dataset: str = "RSLab-Eval"
    n_users: int = 0
    n_items: int = 0
    n_interactions: int = 0
    k_core: int = 0
    split_method: str = "global_temporal"    # random | global_temporal | per_user_temporal | leave_one_out
    train_frac: float = 0.8
    relevance_threshold: str = "rating >= 4"
    K: int = 10
    candidate_set: str = "full_catalog"      # full_catalog | sampled_negatives
    n_negatives: int | None = None
    negative_sampling: str | None = None     # uniform | popularity | in-category
    gain_type: str = "linear"
    seeds: tuple = (0, 1, 2, 3, 4)
    primary_metric: str = "nDCG@10"
    guardrail_metrics: tuple = ("Coverage@10", "Gini", "Novelty", "CalibErr")
    notes: str = ""

    def validate(self):
        problems = []
        if self.candidate_set == "sampled_negatives":
            if not self.n_negatives:
                problems.append("ใช้การสุ่มรายการเชิงลบ แต่ไม่ได้ระบุจำนวน n_negatives")
            if not self.negative_sampling:
                problems.append("ใช้การสุ่มรายการเชิงลบ แต่ไม่ได้ระบุวิธีสุ่ม")
        if self.split_method == "random":
            problems.append("ใช้การแบ่งแบบสุ่ม ซึ่งทำให้อนาคตรั่วเข้าสู่ชุดฝึก")
        if len(self.seeds) < 3:
            problems.append("ควรรันอย่างน้อย 3–5 เมล็ดสุ่ม เพื่อรายงานความผันผวน")
        if not self.primary_metric:
            problems.append("ไม่ได้ประกาศตัวชี้วัดหลัก (Primary Metric)")
        return problems

    def metric_name(self) -> str:
        """ชื่อตัวชี้วัดที่ *ซื่อสัตย์* — ถ้าใช้การสุ่ม ต้องมีคำว่า sampled นำหน้า"""
        if self.candidate_set == "sampled_negatives":
            return f"sampled {self.primary_metric} (m={self.n_negatives}, {self.negative_sampling})"
        return self.primary_metric

    def render(self) -> str:
        lines = ["=" * 66, "บัตรโปรโตคอลการประเมิน (Protocol Card)", "=" * 66]
        for k, v in asdict(self).items():
            if v is None or v == "":
                continue
            lines.append(f"  {k:22s} : {v}")
        lines.append(f"  {'ชื่อตัวชี้วัดที่ต้องรายงาน':22s} : {self.metric_name()}")
        probs = self.validate()
        lines.append("-" * 66)
        if probs:
            lines.append("  ⚠ ข้อบกพร่องที่ต้องแก้ก่อนเผยแพร่:")
            for p in probs:
                lines.append(f"     - {p}")
        else:
            lines.append("  ✓ บัตรนี้ครบถ้วนพอให้ผู้อื่นทำซ้ำได้")
        lines.append("=" * 66)
        return "\n".join(lines)


def build_candidates(target_items, all_items, seen_items, card: ProtocolCard,
                     popularity=None, rng=None):
    """สร้างเซตผู้สมัครตามบัตรโปรโตคอล (รายการเป้าหมายต้องอยู่ในเซตเสมอ)"""
    rng = rng or np.random.default_rng(0)
    pool = [i for i in all_items if i not in seen_items]
    if card.candidate_set == "full_catalog":
        return pool
    m = int(card.n_negatives or 100)
    negs = [i for i in pool if i not in set(target_items)]
    if card.negative_sampling == "popularity" and popularity is not None:
        w = np.array([popularity.get(i, 1) for i in negs], dtype=float)
        w = w / w.sum()
        pick = rng.choice(len(negs), size=min(m, len(negs)), replace=False, p=w)
    else:
        pick = rng.choice(len(negs), size=min(m, len(negs)), replace=False)
    return list(target_items) + [negs[j] for j in pick]


print("✓ เซลล์ 0.4 พร้อม : random_split, global_temporal_split, per_user_temporal_split,")
print("                  leave_one_out_split, leakage_report, k_core_filter,")
print("                  paired_bootstrap, wilcoxon_signed_rank, holm_correction,")
print("                  seed_variance, ProtocolCard, build_candidates")

---
## ส่วนที่ 0.5 — สร้างชุดข้อมูลทั้งหมด (เทียบเท่า `make_dataset.py`)

สมุดนี้ใช้ชุดข้อมูลสองระดับ ซึ่ง **สร้างขึ้นในหน่วยความจำ** ไม่ต้องดาวน์โหลดจากที่ใด

**1. MovieMini-12 — เล็กพอที่จะคำนวณด้วยมือได้ทุกตัวชี้วัด**
ภาพยนตร์ 12 เรื่อง × ผู้ใช้ 8 คน ชุดเดียวกับบทที่ 1 จึงเปรียบเทียบข้ามบทได้
ใช้ในส่วนที่ 1, 2 และ 5

**2. RSLab-Eval — ใหญ่พอที่จะเห็นปรากฏการณ์เชิงสถิติ**
ผู้ใช้ 800 คน × รายการ 500 ชิ้น × 30 สัปดาห์ ออกแบบให้มีสมบัติสามข้อที่บทนี้ต้องใช้

| สมบัติ | ใช้สาธิตเรื่อง |
|---|---|
| ความนิยมกระจายแบบหางยาว (power law) | Coverage · Novelty · Gini |
| ความนิยมเลื่อนตามเวลา (popularity drift) | การรั่วไหลจากการแบ่งแบบสุ่ม |
| หมวดหมู่ชัดเจน 6 หมวด | Diversity · Calibration |

> ทุกอย่างผูกกับ `SEED = 2569` ผลลัพธ์จึงเหมือนกันทุกครั้งที่รัน — นี่คือเงื่อนไขแรกของการทำซ้ำได้

In [ ]:
# ============================================================================
# เซลล์ 0.5  สร้างชุดข้อมูลทั้งหมด (รวมเนื้อหาของ make_dataset.py)
# ผลลัพธ์เก็บไว้ในตัวแปรส่วนกลาง ไม่ต้องอ่านไฟล์จากดิสก์
# ============================================================================

# ---------------------------------------------------- 1) MovieMini-12 : คะแนน
def make_ratings_small() -> pd.DataFrame:
    """คะแนน 1–5 ของผู้ใช้ 8 คน × ภาพยนตร์ 12 เรื่อง (NaN = ยังไม่เคยให้คะแนน)"""
    N = np.nan
    users = ["Emma", "James", "Olivia", "Michael", "Sophia", "Daniel", "Ava", "Ethan"]
    items = ["The Matrix", "Inception", "Interstellar", "Titanic", "La La Land",
             "The Notebook", "John Wick", "Mad Max", "Parasite", "Toy Story",
             "Coco", "Spirited Away"]
    R = np.array([
        [5, 5, 4, 1, N, 1, 4, N, 4, N, N, 3],
        [4, 5, 5, N, 2, N, 3, 4, N, 2, N, N],
        [1, 2, N, 5, 5, 4, N, N, 3, N, 4, 4],
        [N, 3, 4, 2, N, N, 5, 5, 4, N, N, N],
        [2, N, 1, 5, 5, 5, N, 1, N, 4, 5, N],
        [N, 4, 5, 3, 3, N, 2, N, 5, 4, 4, 5],
        [3, 4, N, 4, N, 3, N, N, 5, N, 3, 4],
        [4, N, 3, N, 4, N, 5, 4, N, 5, N, 5]], dtype=float)
    rows = []
    for ui, u in enumerate(users):
        for ii, it in enumerate(items):
            if not np.isnan(R[ui, ii]):
                rows.append({"user_id": f"u{ui+1}", "user_name": u,
                             "item_id": f"i{ii+1}", "item_name": items[ii],
                             "rating": R[ui, ii]})
    return pd.DataFrame(rows)


# ------------------------------------------------ 1b) MovieMini-12 : เมทาดาทา
def make_movie_meta() -> pd.DataFrame:
    """แนว · ค่ายผู้ผลิต · จำนวนผู้เคยชม (จากฐานผู้ใช้ 500 คน)

    ตั้งใจให้ "ค่าย ง" เป็นค่ายเล็กที่ไม่มีเรื่องใดติดห้าอันดับความนิยมแรกเลย
    และให้ห้าอันดับแรกคาบเกี่ยวหลายแนว เพื่อให้เห็นว่าระบบยอดนิยมได้ ILD สูงโดยบังเอิญ
    """
    rows = [
        ("i1",  "The Matrix",    "ไซไฟ",      "ค่าย ก", 400),
        ("i2",  "Inception",     "ไซไฟ",      "ค่าย ก", 150),
        ("i3",  "Interstellar",  "ไซไฟ",      "ค่าย ก",  25),
        ("i4",  "Titanic",       "โรแมนติก",  "ค่าย ข", 300),
        ("i5",  "La La Land",    "โรแมนติก",  "ค่าย ข", 100),
        ("i6",  "The Notebook",  "โรแมนติก",  "ค่าย ง",   8),
        ("i7",  "John Wick",     "แอ็กชัน",   "ค่าย ค", 250),
        ("i8",  "Mad Max",       "แอ็กชัน",   "ค่าย ง",  15),
        ("i9",  "Parasite",      "ระทึกขวัญ", "ค่าย ข",  60),
        ("i10", "Toy Story",     "แอนิเมชัน", "ค่าย ข", 200),
        ("i11", "Coco",          "แอนิเมชัน", "ค่าย ค",  40),
        ("i12", "Spirited Away", "แอนิเมชัน", "ค่าย ง",   2),
    ]
    return pd.DataFrame(rows, columns=["item_id", "item_name", "genre", "studio", "viewers"])


# ------------------------------------------- 2) ลิสต์อันดับของ Model A และ B
def make_ranking_demo() -> pd.DataFrame:
    """Model A และ B มีของถูก *จำนวนเท่ากัน* (3 ชิ้น) แต่วางคนละตำแหน่ง

    A วางไว้อันดับ 1, 3, 5   ส่วน B วางไว้อันดับ 4, 6, 9
    ตัวชี้วัดที่ไวต่อตำแหน่งเท่านั้นที่จะแยกสองระบบนี้ออก
    """
    relevant = {"i2", "i5", "i9"}
    rank_A = ["i2", "i7", "i5", "i3", "i9", "i1", "i4", "i6", "i8", "i10"]
    rank_B = ["i7", "i3", "i1", "i2", "i4", "i5", "i6", "i8", "i9", "i10"]
    rows = []
    for m, r in (("A", rank_A), ("B", rank_B)):
        for pos, it in enumerate(r, 1):
            rows.append({"model": m, "position": pos, "item_id": it,
                         "relevant": int(it in relevant)})
    return pd.DataFrame(rows)


# ------------------------------------------------------------ 3) RSLab-Eval
def make_interactions(n_users=800, n_items=500, n_weeks=30, seed=SEED):
    """ปฏิสัมพันธ์สังเคราะห์ที่มีหางยาวและมีการเลื่อนของความนิยมตามเวลา

    * ความนิยมพื้นฐานดึงจากการแจกแจงกฎกำลัง (Pareto alpha = 1.6)
    * รายการแบ่ง 6 หมวด ผู้ใช้แต่ละคนมีหมวดโปรด 1–2 หมวด
    * ความนิยมเลื่อนทุกสัปดาห์ (บางเรื่องมาแรง บางเรื่องซาลง)
      กลไกนี้เองที่ทำให้การแบ่งข้อมูลแบบสุ่มให้ผลดีเกินจริง
    """
    rng = np.random.default_rng(seed)
    items = np.array([f"it{n:03d}" for n in range(n_items)])
    cats = np.array([f"c{n % 6}" for n in range(n_items)])
    providers = np.array([f"p{n % 25}" for n in range(n_items)])
    cat_idx = np.array([int(c[1]) for c in cats])

    base_pop = rng.pareto(1.6, n_items) + 1
    base_pop = base_pop / base_pop.sum()
    drift = rng.normal(0.0, 0.9, n_items)
    user_fav = [rng.choice(6, size=rng.integers(1, 3), replace=False) for _ in range(n_users)]

    rows = []
    for w in range(n_weeks):
        t_norm = w / max(1, n_weeks - 1)
        pop_w = base_pop * np.exp(drift * t_norm)
        pop_w = pop_w / pop_w.sum()
        n_events = rng.integers(700, 1000)
        for _ in range(n_events):
            u = rng.integers(0, n_users)
            in_fav = np.isin(cat_idx, user_fav[u])
            w_u = pop_w.copy()
            w_u[in_fav] *= 6.0                       # ผู้ใช้เอนเอียงไปหาหมวดโปรด
            w_u = w_u / w_u.sum()
            i = rng.choice(n_items, p=w_u)
            ts = w * 7 * 86400 + float(rng.uniform(0, 7 * 86400))
            mu = 4.1 if in_fav[i] else 2.9           # หมวดโปรดมักได้คะแนนสูงกว่า
            rating = float(np.clip(round(rng.normal(mu, 0.9)), 1, 5))
            rows.append({"user_id": f"u{u:03d}", "item_id": items[i],
                         "rating": rating, "timestamp": ts, "week": w})
    df = (pd.DataFrame(rows).sort_values("timestamp")
          .drop_duplicates(subset=["user_id", "item_id"], keep="first")
          .reset_index(drop=True))
    meta = pd.DataFrame({"item_id": items, "category": cats, "provider": providers,
                         "base_popularity": base_pop, "drift": drift})
    return df, meta


# ---------------------------------------------------------------- สร้างจริง
t0 = time.time()
ratings_small = make_ratings_small()
movie_meta = make_movie_meta()
ranking_demo = make_ranking_demo()
interactions, item_meta = make_interactions()

# พจนานุกรมที่ใช้ซ้ำตลอดสมุด
MM_IDS = list(movie_meta.item_id)
MM_TITLE = dict(zip(movie_meta.item_id, movie_meta.item_name))
MM_GENRE = dict(zip(movie_meta.item_id, movie_meta.genre))
MM_STUDIO = dict(zip(movie_meta.item_id, movie_meta.studio))
MM_POP = dict(zip(movie_meta.item_id, movie_meta.viewers))
MM_PLATFORM_USERS = 500          # ฐานผู้ใช้ทั้งแพลตฟอร์ม ใช้คำนวณ p(i) ของ Novelty

ALL_ITEMS = sorted(interactions.item_id.unique())
CAT_OF = dict(zip(item_meta.item_id, item_meta.category))
PROV_OF = dict(zip(item_meta.item_id, item_meta.provider))

print(f"สร้างข้อมูลเสร็จใน {time.time()-t0:.1f} วินาที\n")
print(f"[1] ratings_small : {len(ratings_small)} คะแนน "
      f"({ratings_small.user_id.nunique()} ผู้ใช้ × {ratings_small.item_id.nunique()} เรื่อง)")
print(f"[2] movie_meta    : {len(movie_meta)} เรื่อง "
      f"({movie_meta.genre.nunique()} แนว, {movie_meta.studio.nunique()} ค่าย)")
print(f"[3] ranking_demo  : {len(ranking_demo)} แถว (2 แบบจำลอง × 10 ตำแหน่ง)")
print(f"[4] interactions  : {len(interactions):,} ปฏิสัมพันธ์ "
      f"({interactions.user_id.nunique()} ผู้ใช้ × {interactions.item_id.nunique()} รายการ) "
      f"ความหนาแน่น {len(interactions)/(interactions.user_id.nunique()*500):.4f}")
print(f"[5] item_meta     : {len(item_meta)} รายการ "
      f"({item_meta.category.nunique()} หมวด, {item_meta.provider.nunique()} ผู้ผลิต)")
print(f"    ช่วงเวลา     : สัปดาห์ที่ {interactions.week.min()} ถึง {interactions.week.max()}")

display(ratings_small.head(4))
display(movie_meta.head(4))

---
# ส่วนที่ 1 — MAE และ RMSE
### (เทียบเท่า `lab2_01_rating_metrics.py` · ตำราหัวข้อ 2.1)

**คำถามวิจัย** MAE กับ RMSE ต่างกันอย่างไร และความต่างนั้นสำคัญเมื่อใด

$$\text{MAE}=\frac{1}{|\mathcal{T}|}\sum |\hat r_{ui}-r_{ui}| \qquad
\text{RMSE}=\sqrt{\frac{1}{|\mathcal{T}|}\sum (\hat r_{ui}-r_{ui})^2}$$

**สิ่งที่ต้องสังเกต**

1. ตารางความคลาดเคลื่อนทีละแถว — ตรวจการคำนวณด้วยมือได้
2. กรณีที่ **MAE เท่ากันเป๊ะ แต่ RMSE ต่างกันสองเท่า**
3. การพิสูจน์เชิงตัวเลขว่า `RMSE ≥ MAE` เสมอ (ผลจากอสมการของเยนเซน)

In [ ]:
# ============================================================================
# การทดลองที่ 2.1  ตัวชี้วัดการพยากรณ์คะแนน  (lab2_01_rating_metrics.py)
# ============================================================================
def show_error_table(y_true, y_pred, labels, title):
    print(f"\n{title}")
    print("-" * 74)
    print(f"{'รายการ':<16}{'จริง':>8}{'ทาย':>10}{'ผิดพลาด':>12}{'|ผิดพลาด|':>14}{'ผิดพลาด²':>14}")
    print("-" * 74)
    rows = error_table(y_true, y_pred, labels)
    for r in rows:
        print(f"{r['label']:<16}{r['y_true']:>8.1f}{r['y_pred']:>10.2f}"
              f"{r['error']:>12.2f}{r['abs_error']:>14.2f}{r['sq_error']:>14.4f}")
    s_abs = sum(r["abs_error"] for r in rows)
    s_sq = sum(r["sq_error"] for r in rows)
    n = len(rows)
    print("-" * 74)
    print(f"{'ผลรวม':<16}{'':>8}{'':>10}{'':>12}{s_abs:>14.2f}{s_sq:>14.4f}")
    print(f"\n  MAE  = {s_abs:.2f} / {n} = {mae(y_true, y_pred):.4f}")
    print(f"  RMSE = sqrt({s_sq:.4f} / {n}) = sqrt({s_sq/n:.4f}) = {rmse(y_true, y_pred):.4f}")
    return mae(y_true, y_pred), rmse(y_true, y_pred)


def exp_2_1():
    print("=" * 74)
    print("การทดลองที่ 2.1  ตัวชี้วัดการพยากรณ์คะแนน")
    print("=" * 74)

    # ---- ส่วนที่ 1 ตัวอย่างหลัก (ตำราตัวอย่าง 2.1)
    labels = ["The Matrix", "Inception", "Interstellar", "Titanic", "The Notebook"]
    y_true = [5, 5, 4, 1, 1]
    y_pred = [4.2, 4.6, 4.3, 2.4, 1.5]
    m1, r1 = show_error_table(y_true, y_pred, labels,
                              "ส่วนที่ 1  Model A บนคะแนนจริงของผู้ใช้ Emma")
    print(f"\n  ตีความ : โดยเฉลี่ยทายพลาด {m1:.4f} ดาว เมื่อถ่วงน้ำหนักกรณีพลาดหนัก "
          f"ความคลาดเคลื่อนอยู่ที่ {r1:.4f} ดาว")
    print(f"  ช่องว่าง RMSE − MAE = {r1-m1:.4f} คือสัญญาณของ 'ความไม่สม่ำเสมอของความผิดพลาด'")
    print("  ในที่นี้มาจาก Titanic ที่ทายพลาดถึง 1.40 ดาวเพียงเรื่องเดียว")

    # ---- ส่วนที่ 2 MAE เท่ากัน แต่ RMSE ต่างกัน (ตำราตัวอย่าง 2.2)
    print("\n" + "=" * 74)
    print("ส่วนที่ 2  กรณีที่ MAE เท่ากันทุกประการ แต่ RMSE ต่างกันชัดเจน")
    print("=" * 74)
    yt = [4, 4, 4, 4]
    pred_even = [3.0, 3.0, 3.0, 3.0]      # ผิดทีละ 1.0 ทุกตัว
    pred_spiky = [4.0, 4.0, 4.0, 0.0]     # ถูกสามตัว แต่พลาด 4.0 หนึ่งตัว
    for name, yp in [("กระจายสม่ำเสมอ", pred_even), ("พลาดหนักจุดเดียว", pred_spiky)]:
        print(f"\n  {name}: ทาย = {yp}")
        print(f"     MAE  = {mae(yt, yp):.4f}")
        print(f"     RMSE = {rmse(yt, yp):.4f}")
    print(f"\n  ข้อสรุป: MAE = 1.0000 เท่ากันทั้งสองกรณี แต่ RMSE ต่างกัน "
          f"{rmse(yt, pred_spiky)/rmse(yt, pred_even):.1f} เท่า")
    print("  MAE มองไม่เห็นความต่าง เพราะพิจารณาเพียงขนาดความผิดพลาดเฉลี่ย")
    print("  RMSE ยกกำลังสองก่อนเฉลี่ย จึงชี้ว่ากรณีที่ 2 มีความผิดพลาดขนาดใหญ่")
    print("  -> ต้องรายงานทั้งสองค่าเสมอ")

    # ---- ส่วนที่ 3 ตรวจสอบอสมการ RMSE >= MAE
    print("\n" + "=" * 74)
    print("ส่วนที่ 3  ตรวจสอบเชิงตัวเลขว่า RMSE ≥ MAE เสมอ (สุ่ม 10,000 กรณี)")
    print("=" * 74)
    rng = np.random.default_rng(0)
    worst = 1e9
    for _ in range(10000):
        n = rng.integers(2, 40)
        worst = min(worst, rmse(rng.uniform(1, 5, n), rng.uniform(1, 5, n))
                    - mae(rng.uniform(1, 5, n), rng.uniform(1, 5, n)))
    a = rng.uniform(1, 5, (10000, 12))
    b = rng.uniform(1, 5, (10000, 12))
    d = np.sqrt(((a - b) ** 2).mean(axis=1)) - np.abs(a - b).mean(axis=1)
    print(f"  ค่าต่ำสุดของ (RMSE − MAE) จากการสุ่ม 10,000 กรณี = {d.min():.6f}  -> ไม่เคยติดลบ")
    print("  เหตุผล: อสมการของเยนเซน (Jensen's inequality) เพราะการยกกำลังสองเป็นฟังก์ชันคอนเวกซ์")

    # ---- กราฟสรุป
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].bar(["uniform\nerrors", "one big\nerror"],
              [mae(yt, pred_even), mae(yt, pred_spiky)], color="#4C78A8", label="MAE")
    ax[0].bar(["uniform\nerrors", "one big\nerror"],
              [rmse(yt, pred_even), rmse(yt, pred_spiky)], color="#F58518",
              width=0.45, label="RMSE")
    ax[0].set_title("Same MAE, different RMSE")
    ax[0].set_ylabel("error (stars)")
    ax[0].legend()
    ax[1].hist(d, bins=60, color="#54A24B")
    ax[1].axvline(0, color="crimson", lw=1.5)
    ax[1].set_title("RMSE − MAE over 10,000 random cases")
    ax[1].set_xlabel("RMSE − MAE")
    plt.tight_layout()
    plt.show()


exp_2_1()

In [ ]:
# ---------------------------------------------------------------------------
# ทดลองเอง 1 : เปลี่ยนค่าพยากรณ์แล้วดูว่า MAE กับ RMSE ตอบสนองต่างกันอย่างไร
# ---------------------------------------------------------------------------
y_true = [5, 5, 4, 1, 1]
y_pred = [4.2, 4.6, 4.3, 2.4, 1.5]        # <-- ลองแก้ตัวเลขในบรรทัดนี้

print(f"MAE      = {mae(y_true, y_pred):.4f}")
print(f"RMSE     = {rmse(y_true, y_pred):.4f}")
print(f"RMSE/MAE = {rmse(y_true, y_pred)/mae(y_true, y_pred):.4f}   (≥ 1 เสมอ)")
print("\nยิ่งอัตราส่วน RMSE/MAE สูง ยิ่งแปลว่าความผิดพลาดกระจุกอยู่ที่ไม่กี่รายการ")

---
# ส่วนที่ 2 — ตัวชี้วัดเชิงอันดับแปดตัว
### (เทียบเท่า `lab2_02_ranking_metrics.py` · ตำราหัวข้อ 2.2–2.4)

**คำถามวิจัย** แบบจำลองสองตัวที่หยิบของถูก *จำนวนเท่ากัน* แต่วางตำแหน่งต่างกัน
ตัวชี้วัดตัวใดแยกความต่างนี้ออก และตัวใดแยกไม่ออกเลย

**ข้อมูลตั้งต้น** (ตารางที่ 2.2 ในตำรา) Ground Truth = `{i2, i5, i9}` · K = 10

| | อันดับของรายการที่ถูก |
|---|---|
| Model A | 1, 3, 5 |
| Model B | 4, 6, 9 |

**คำตอบที่จะได้เห็น**

* `P@K`, `R@K`, `F1@K`, `HR@K` — **แยกไม่ออกเลย** เพราะตรวจเพียงว่า "อยู่ใน Top-10 หรือไม่"
* `RR/MRR`, `AP/MAP`, `nDCG`, `AUC` — **แยกออกชัดเจน** และขนาดความต่างก็ไม่เท่ากัน (1.73 ถึง 4.00 เท่า)

> **บทเรียนสำคัญ** การเลือกตัวชี้วัดคือการเลือกข้อสรุป ทีมที่รายงาน `Precision@10` จะสรุปว่าเท่ากัน
> ทีมที่รายงาน `MRR` จะสรุปว่า A ดีกว่าสี่เท่า — ทั้งสองทีมคำนวณถูกทุกหลัก
> จึงต้อง **ประกาศ Primary Metric ไว้ก่อนเห็นผล**

In [ ]:
# ============================================================================
# การทดลองที่ 2.2  ตัวชี้วัดเชิงอันดับแปดตัว  (lab2_02_ranking_metrics.py)
# ============================================================================
def trace_table(ranked, relevant, K):
    """ตารางสะสมทีละตำแหน่ง เพื่อให้ตรวจการคำนวณด้วยมือได้"""
    print(f"{'ตำแหน่ง':>8}{'รายการ':>10}{'rel':>6}{'Precision ณ จุดนี้':>22}"
          f"{'ส่วนลด 1/log2(p+1)':>22}{'ส่วนเพิ่ม DCG':>16}")
    print("-" * 86)
    hits, dcg, ap_terms = 0, 0.0, []
    for p, it in enumerate(ranked[:K], 1):
        rel = 1 if it in relevant else 0
        disc = 1.0 / np.log2(p + 1)
        add = rel * disc
        dcg += add
        if rel:
            hits += 1
            ap_terms.append(hits / p)
            pstr = f"{hits}/{p} = {hits/p:.4f}"
        else:
            pstr = "—"
        print(f"{p:>8}{it:>10}{rel:>6}{pstr:>22}{disc:>22.4f}{add:>16.4f}")
    print("-" * 86)
    return dcg, ap_terms


def exp_2_2():
    print("=" * 86)
    print("การทดลองที่ 2.2  ตัวชี้วัดเชิงอันดับแปดตัวบนรายการแนะนำชุดเดียวกัน")
    print("=" * 86)

    df = ranking_demo
    relevant = set(df.loc[df.relevant == 1, "item_id"].unique())
    K = 10
    print(f"\nGround Truth = {sorted(relevant)}  ทั้งหมด {len(relevant)} รายการ   ค่าตัด K = {K}")

    ranks = {}
    for m in ["A", "B"]:
        ranks[m] = list(df[df.model == m].sort_values("position").item_id)
        pos = [i + 1 for i, it in enumerate(ranks[m]) if it in relevant]
        print(f"\nModel {m} : {ranks[m]}")
        print(f"          รายการที่ถูกอยู่อันดับ {pos}")

    # ---- ส่วนที่ 1 ตารางสะสมทีละตำแหน่ง
    for m in ["A", "B"]:
        print("\n" + "=" * 86)
        print(f"ส่วนที่ 1  ตารางสะสมของ Model {m}")
        print("=" * 86)
        dcg, ap_terms = trace_table(ranks[m], relevant, K)
        idcg = dcg_at_k([1.0] * len(relevant), K)
        print(f"  DCG@{K}  = {dcg:.4f}")
        print(f"  IDCG@{K} = {idcg:.4f}   (ลิสต์อุดมคติ: rel = 1 ที่ตำแหน่ง 1..{len(relevant)})")
        print(f"  nDCG@{K} = {dcg:.4f} / {idcg:.4f} = {dcg/idcg:.4f}")
        print(f"  AP@{K}   = ({' + '.join(f'{a:.4f}' for a in ap_terms)}) / {len(relevant)}"
              f" = {sum(ap_terms)/len(relevant):.4f}")

    # ---- ส่วนที่ 2 ตารางเปรียบเทียบแปดตัว
    print("\n" + "=" * 86)
    print("ส่วนที่ 2  ตารางเปรียบเทียบตัวชี้วัดทั้งแปดตัว (ตารางที่ 2.3 ในตำรา)")
    print("=" * 86)
    rel_map = {it: 1.0 for it in relevant}
    comp = pd.DataFrame([{"Model": m, **{k: round(v, 4)
                                         for k, v in evaluate_user(ranks[m], rel_map, K).items()}}
                         for m in ["A", "B"]]).set_index("Model")
    display(comp)

    same = [c for c in comp.columns if abs(comp.loc["A", c] - comp.loc["B", c]) < 1e-9]
    diff = [c for c in comp.columns if c not in same]
    print(f"\n  ตัวชี้วัดที่ *แยกไม่ออก* : {same}")
    print("     -> เพราะนับเพียง 'จำนวน' ของถูก ไม่สนใจตำแหน่ง")
    print("\n  ตัวชี้วัดที่ *แยกออก* และขนาดของความต่าง")
    for c in diff:
        a, b = comp.loc["A", c], comp.loc["B", c]
        print(f"     {c:<8} A = {a:.4f}  B = {b:.4f}   A/B = {a/b if b else float('inf'):.2f} เท่า")
    print("\n  ข้อสังเกต: แม้ในกลุ่มที่แยกออกได้ อัตราส่วน A/B ยังต่างกันตั้งแต่ 1.73 ถึง 4.00 เท่า")
    print("            การอ้างว่า 'ดีกว่าอย่างมีนัยสำคัญ' จึงไม่มีความหมายหากไม่ระบุตัวชี้วัด")

    # ---- ส่วนที่ 3 พฤติกรรมเมื่อ K เปลี่ยน
    print("\n" + "=" * 86)
    print("ส่วนที่ 3  พฤติกรรมของตัวชี้วัดเมื่อค่าตัด K เปลี่ยน")
    print("=" * 86)
    rows = []
    for k in range(1, 11):
        for m in ["A", "B"]:
            rows.append({"K": k, "Model": m,
                         "P@K": precision_at_k(ranks[m], relevant, k),
                         "R@K": recall_at_k(ranks[m], relevant, k),
                         "F1@K": f1_at_k(ranks[m], relevant, k),
                         "nDCG@K": ndcg_at_k(ranks[m], rel_map, k)})
    sweep = pd.DataFrame(rows)
    display(sweep[sweep.Model == "A"].set_index("K").drop(columns="Model").round(4))

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
    for ax, col in zip(axes, ["P@K", "R@K", "nDCG@K"]):
        for m, c in (("A", "#4C78A8"), ("B", "#F58518")):
            s = sweep[sweep.Model == m]
            ax.plot(s.K, s[col], marker="o", ms=4, color=c, label=f"Model {m}")
        ax.set_title(col)
        ax.set_xlabel("K")
        ax.set_ylim(-0.05, 1.05)
    axes[0].legend()
    plt.tight_layout()
    plt.show()

    print("  ข้อสังเกตเชิงโครงสร้าง (จริงเสมอ ไม่ใช่ความบังเอิญของข้อมูลชุดนี้)")
    print("     * Recall@K ไม่มีวันลดลงเมื่อ K เพิ่ม  (ตัวเศษเพิ่มได้ ตัวส่วนคงที่)")
    print("     * Precision@K มีแนวโน้มลดลงเมื่อ K เพิ่ม (ตัวส่วนโตเป็นเชิงเส้น)")
    print("     * จึง 'ห้ามเปรียบเทียบค่าที่ K ต่างกัน' โดยเด็ดขาด")
    return comp


comp_2_2 = exp_2_2()

In [ ]:
# ---------------------------------------------------------------------------
# ทดลองเอง 2 : สลับตำแหน่งของรายการที่เกี่ยวข้อง แล้วดูว่าตัวชี้วัดใดเปลี่ยน
# ---------------------------------------------------------------------------
relevant = {"i2", "i5", "i9"}
my_rank = ["i2", "i5", "i9", "i1", "i3", "i4", "i6", "i7", "i8", "i10"]   # <-- ลองสลับดู

res = evaluate_user(my_rank, {i: 1.0 for i in relevant}, k=10)
display(pd.DataFrame([res], index=["ลิสต์ของคุณ"]).round(4))

pos = [i + 1 for i, it in enumerate(my_rank) if it in relevant]
print(f"รายการที่ถูกอยู่อันดับ {pos}")
print("ลองเลื่อนของถูกไปไว้ท้ายลิสต์ แล้วสังเกตว่า P@10/R@10/F1/HR ไม่ขยับเลย")
print("ขณะที่ RR, AP@10, nDCG@10, AUC ลดลงทันที")

---
# ส่วนที่ 3 — การแบ่งข้อมูลตามเวลาและข้อมูลรั่วไหล
### (เทียบเท่า `lab2_03_leakage_temporal.py` · ตำราหัวข้อ 2.8)

**คำถามวิจัย** การแบ่งข้อมูลแบบสุ่มทำให้ตัวเลขที่รายงานสูงเกินจริงเท่าใด และเพราะอะไร

**เงื่อนไขการทดลอง** แบบจำลอง `TopPopular` ตัวเดิมทุกประการ ข้อมูลเดิม ตัวชี้วัดเดิม
**เปลี่ยนเพียงวิธีแบ่งข้อมูลเท่านั้น** ความต่างของตัวเลขจึงมาจากวิธีแบ่งล้วน ๆ

| วิธีแบ่ง | หลักการ | ปัญหา |
|---|---|---|
| Random | สุ่ม 20% ไปทดสอบ | ใช้อนาคตทำนายอดีต |
| Global Temporal | จุดตัดเวลาจุดเดียวทั้งระบบ | สะท้อนการใช้งานจริงที่สุด |
| Per-User Temporal | กันส่วนท้าย 20% ของแต่ละคน | ยังรั่วข้ามผู้ใช้ |
| Leave-One-Out | กันเหตุการณ์ล่าสุดคนละหนึ่งชิ้น | ความแปรปรวนสูงมาก |

**กลไกเบื้องหลัง** *Popularity Drift* — เมื่อแบ่งแบบสุ่ม `TopPopular` ได้เห็นความนิยม
"ในสัปดาห์เดียวกับที่ต้องทำนาย" จึงแม่นผิดปกติ สมบัตินี้มีในข้อมูลจริงทุกชุด

> **วินัยเดียวที่ป้องกันได้** การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ — ก่อนคำนวณสถิติใด ๆ
> แม้แต่ค่าเฉลี่ยหรือพจนานุกรมรายการ

In [ ]:
# ============================================================================
# การทดลองที่ 2.3  การแบ่งข้อมูลตามเวลาและข้อมูลรั่วไหล  (lab2_03)
# ============================================================================
K = 10
REL_THRESHOLD = 4.0


def popularity_ranker(train_df):
    """TopPopular — จัดอันดับตามความนิยมในชุดฝึกเท่านั้น

    เลือกแบบจำลองที่ง่ายที่สุดโดยเจตนา เพื่อให้แน่ใจว่าความต่างของตัวเลข
    มาจาก *วิธีแบ่งข้อมูล* ล้วน ๆ ไม่ได้มาจากความสามารถของแบบจำลอง
    """
    pop = Counter(train_df.item_id)
    order = [it for it, _ in pop.most_common()]

    def rank(user, candidates):
        cset = set(candidates)
        head = [it for it in order if it in cset]
        hset = set(head)
        return head + [it for it in candidates if it not in hset]
    return rank


def evaluate_split(rank_fn, train, test, all_items, K=K):
    """ประเมินด้วยการจัดอันดับเต็มคลัง (ตัดรายการที่ผู้ใช้เห็นแล้วในชุดฝึกออก)"""
    seen = train.groupby("user_id")["item_id"].apply(set).to_dict()
    rel = (test[test.rating >= REL_THRESHOLD]
           .groupby("user_id")["item_id"].apply(set).to_dict())
    scores = []
    for u, items in rel.items():
        cands = [i for i in all_items if i not in seen.get(u, set())]
        if not items & set(cands):
            continue
        scores.append(ndcg_at_k(rank_fn(u, cands), {i: 1.0 for i in items}, K))
    return (float(np.mean(scores)) if scores else float("nan")), len(scores)


def exp_2_3():
    df = interactions
    all_items = ALL_ITEMS
    print("=" * 84)
    print("การทดลองที่ 2.3  การแบ่งข้อมูลตามเวลาและข้อมูลรั่วไหล")
    print("=" * 84)
    print(f"\nRSLab-Eval : ปฏิสัมพันธ์ {len(df):,}  ผู้ใช้ {df.user_id.nunique():,}"
          f"  รายการ {df.item_id.nunique():,}")
    print(f"ช่วงเวลา : สัปดาห์ที่ {df.week.min()} ถึง {df.week.max()}")
    print(f"เกณฑ์ความเกี่ยวข้อง : rating >= {REL_THRESHOLD}")

    splits = {}
    tr, te = random_split(df, 0.2, seed=42)
    splits["สุ่ม (Random)"] = (tr, te)
    tr, _, te = global_temporal_split(df, train_frac=0.8)
    splits["ตามเวลาแบบรวม (Global Temporal)"] = (tr, te)
    tr, te = per_user_temporal_split(df, 0.2)
    splits["ตามเวลารายผู้ใช้ (Per-User Temporal)"] = (tr, te)
    tr, te = leave_one_out_split(df)
    splits["กันรายการสุดท้าย (Leave-One-Out)"] = (tr, te)

    # ---- ส่วนที่ 1 รายงานการรั่วไหล
    print("\n" + "=" * 84)
    print("ส่วนที่ 1  ตรวจการรั่วไหลของเวลาในการแบ่งสี่แบบ")
    print("=" * 84)
    print(f"\n{'วิธีแบ่ง':<38}{'ฝึก':>9}{'ทดสอบ':>9}{'อนาคตในชุดฝึก':>20}{'ผลตรวจ':>16}")
    print("-" * 92)
    for name, (a, b) in splits.items():
        rep = leakage_report(a, b)
        print(f"{name:<38}{rep['n_train']:>9,}{rep['n_test']:>9,}"
              f"{rep['n_future_in_train']:>12,} ({rep['pct_future']:>4.1f}%)"
              f"{rep['verdict']:>16}")

    # ---- ส่วนที่ 2 ตัวเลขที่รายงานภายใต้การแบ่งต่างกัน
    print("\n" + "=" * 84)
    print("ส่วนที่ 2  แบบจำลองเดิมทุกประการ แต่วิธีแบ่งต่างกัน (ตารางที่ 2.9)")
    print("=" * 84)
    print(f"\n{'วิธีแบ่ง':<38}{'nDCG@10':>12}{'ผู้ใช้ที่ประเมินได้':>22}{'เทียบตามเวลา':>16}")
    print("-" * 92)
    res = {n: evaluate_split(popularity_ranker(a), a, b, all_items)
           for n, (a, b) in splits.items()}
    base = next(v[0] for k, v in res.items() if "Global Temporal" in k)
    for name, (s, n) in res.items():
        pct = "—" if s == base else f"{(s/base-1)*100:+.1f}%"
        print(f"{name:<38}{s:>12.4f}{n:>22,}{pct:>16}")
    print("\n  ข้อสังเกตสองข้อ")
    print("     1. การแบ่งแบบสุ่มให้ตัวเลขสูงกว่าการแบ่งตามเวลาอย่างเป็นระบบ")
    print("        เพราะแบบจำลองได้เห็น 'ความนิยมในอนาคต' ระหว่างฝึก (popularity drift)")
    print("     2. Leave-One-Out ประเมินได้เพียงบางส่วนของผู้ใช้ทั้งหมด")
    print("        การเทียบค่ากับวิธีอื่นจึงเป็นการเทียบคนละประชากร")
    print("        -> ต้องรายงานจำนวนผู้ใช้ที่ประเมินได้ควบคู่เสมอ")

    # ---- ส่วนที่ 3 การรั่วไหลจากสถิติ
    print("\n" + "=" * 84)
    print("ส่วนที่ 3  การรั่วไหลจากสถิติ (Statistic Leakage) — ตัวอย่างที่ไม่มีบรรทัดใดดูผิดเลย")
    print("=" * 84)
    tr, _, te = global_temporal_split(df, train_frac=0.8)
    s_ok, _ = evaluate_split(popularity_ranker(tr), tr, te, all_items)
    s_bad, _ = evaluate_split(popularity_ranker(df), tr, te, all_items)
    print("\n  ✓ ลำดับที่ถูก : แบ่งข้อมูลก่อน แล้วจึงคำนวณความนิยมจาก 'ชุดฝึก' เท่านั้น")
    print(f"     nDCG@10 = {s_ok:.4f}")
    print("\n  ✗ ลำดับที่ผิด : คำนวณความนิยมจาก 'ข้อมูลทั้งหมด' ก่อนแบ่ง")
    print(f"     nDCG@10 = {s_bad:.4f}")
    print(f"\n  ตัวเลขสูงเกินจริง {(s_bad/s_ok-1)*100:+.1f}%  โดยไม่มีบรรทัดใดในโปรแกรมที่ 'ดูเหมือน' ผิด")
    print("  ส่วนต่างระดับนี้กลบผลของวิธีการใหม่ได้ทั้งหมด (งานตีพิมพ์จำนวนมากอ้างการปรับปรุงเพียง 2–5%)")

    # ---- กราฟ
    names_en = ["Random", "Global\nTemporal", "Per-User\nTemporal", "Leave-\nOne-Out"]
    vals = [v[0] for v in res.values()]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    colors = ["#E45756" if n == "Random" else "#4C78A8" for n in names_en]
    ax[0].bar(names_en, vals, color=colors)
    ax[0].axhline(base, ls="--", color="grey", lw=1)
    ax[0].set_title("Same TopPopular model, 4 split strategies")
    ax[0].set_ylabel("nDCG@10")
    for i, v in enumerate(vals):
        ax[0].text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    ax[1].bar(["correct\n(train only)", "leaked\n(all data)"], [s_ok, s_bad],
              color=["#54A24B", "#E45756"])
    ax[1].set_title("Statistic leakage: popularity computed before split")
    ax[1].set_ylabel("nDCG@10")
    for i, v in enumerate([s_ok, s_bad]):
        ax[1].text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()
    return splits


splits_2_3 = exp_2_3()

---
# ส่วนที่ 4 — ฟูลแค็ตตาล็อก เทียบ การสุ่มรายการเชิงลบ ⭐
### (เทียบเท่า `lab2_04_sampled_vs_full.py` · ตำราหัวข้อ 2.9 — หัวข้อที่สำคัญที่สุดของบท)

**คำถามวิจัย** การสุ่มรายการเชิงลบมาไม่กี่สิบชิ้นแทนการจัดอันดับทั้งคลัง
เปลี่ยนแค่ "ตัวเลข" หรือเปลี่ยน "ผู้ชนะ" ด้วย

**สูตรที่ต้องจำ** เมื่อสุ่มรายการเชิงลบ $m$ ชิ้นจากคลังขนาด $|\mathcal{I}|$

$$\mathbb{E}[\tilde{\rho}] \approx 1 + (\rho - 1)\cdot\frac{m}{|\mathcal{I}|-1}$$

การสุ่ม **บีบอันดับทุกตัวเข้าหา 1 ด้วยตัวประกอบเชิงเส้นเท่ากันหมด**

**ข้อคิดที่ผิด** *"ในเมื่อการสุ่มบีบทุกแบบจำลองเท่ากัน การเปรียบเทียบก็น่าจะยังยุติธรรม"*

**เหตุผลที่ผิด — Hard Cutoff ที่ K** `nDCG@K` ไม่ใช่ฟังก์ชันเชิงเส้นของอันดับ:
อันดับ 11 ให้ 0 ขณะที่อันดับ 10 ให้ 0.2891 การประกอบฟังก์ชันไม่เชิงเส้นเข้ากับ
การแปลงเชิงเส้น **ไม่รักษาลำดับของค่าเฉลี่ย** (Krichene & Rendle, KDD 2020)

**สามการทดลองในส่วนนี้**

1. ผู้ใช้คนเดียว รายการเป้าหมายอยู่อันดับ 500 จาก 5,000 → คะแนนกระโดดจาก 0.0000 เป็น ~0.15
2. ตรวจสอบสูตรเชิงทฤษฎีด้วยการจำลอง Monte Carlo
3. **Winner Flip** — Model P เทียบ Model Q ผู้ชนะสลับกันเมื่อเปลี่ยน $m$

In [ ]:
# ============================================================================
# การทดลองที่ 2.4  ฟูลแค็ตตาล็อก เทียบ การสุ่มรายการเชิงลบ  (lab2_04)
# ============================================================================
def ndcg_from_rank(rank: int, K: int = 10) -> float:
    """nDCG@K เมื่อมีรายการเกี่ยวข้องเพียงหนึ่งชิ้น ณ อันดับ rank

    IDCG = 1 (รายการเดียวอยู่อันดับ 1) ดังนั้น nDCG = DCG
    ถ้า rank > K จะได้ 0 ทันที เพราะถูกตัดออกนอก Top-K  <- 'จุดตัดแข็ง'
    """
    return 0.0 if rank > K else 1.0 / np.log2(rank + 1)


def sampled_rank(true_rank: int, n_items: int, m: int, rng) -> int:
    """อันดับที่สังเกตได้เมื่อสุ่มรายการเชิงลบ m ชิ้นแบบสม่ำเสมอ

    รายการที่ 'เหนือกว่า' เป้าหมายมี (true_rank − 1) ชิ้น จากรายการเชิงลบทั้งหมด
    (n_items − 1) ชิ้น จำนวนที่ยังเหนือกว่าหลังสุ่มเป็นตัวแปรสุ่มไฮเพอร์จีออเมตริก
    อันดับที่สังเกตได้ = 1 + (จำนวนที่เหนือกว่าและถูกสุ่มติดมา)
    """
    n_better = true_rank - 1
    n_neg = n_items - 1
    drawn = rng.hypergeometric(ngood=n_better, nbad=n_neg - n_better,
                               nsample=min(m, n_neg))
    return 1 + int(drawn)


def exp_2_4():
    rng = np.random.default_rng(SEED)
    N_ITEMS, K = 5000, 10

    # ---------------- ส่วนที่ 1
    print("=" * 78)
    print("ส่วนที่ 1  ผู้ใช้คนเดียว: รายการที่ผู้ใช้ชอบจริงอยู่อันดับที่ 500 จาก 5,000")
    print("=" * 78)
    TRUE_RANK = 500
    print(f"\n  จัดอันดับเต็มคลัง (Full Catalog)")
    print(f"     อันดับจริง = {TRUE_RANK}  ซึ่ง > K = {K}")
    print(f"     nDCG@{K} = {ndcg_from_rank(TRUE_RANK, K):.4f}   <- ศูนย์สนิท")
    exp_theory = 1 + (TRUE_RANK - 1) * 99 / (N_ITEMS - 1)
    trials = np.array([sampled_rank(TRUE_RANK, N_ITEMS, 99, rng) for _ in range(20000)])
    scores = np.array([ndcg_from_rank(int(r), K) for r in trials])
    print(f"\n  สุ่มรายการเชิงลบ m = 99 (เซตผู้สมัคร 100 ชิ้น)")
    print(f"     อันดับคาดหวังตามทฤษฎี = 1 + (500−1) × 99/4999 = {exp_theory:.2f}")
    print(f"     อันดับคาดหวังจากการจำลอง 20,000 รอบ = {trials.mean():.2f}")
    print(f"     สัดส่วนที่เลื่อนเข้ามาอยู่ใน Top-{K} = {(trials <= K).mean()*100:.1f}%")
    print(f"     sampled nDCG@{K} เฉลี่ย = {scores.mean():.4f}")
    print(f"\n  สรุปที่ต้องอ่านช้า ๆ : ตัวเลขกระโดดจาก 0.0000 เป็น {scores.mean():.4f}")
    print("     โดยแบบจำลอง 'ไม่ได้ดีขึ้นเลยแม้แต่พารามิเตอร์เดียว'")
    print("     สิ่งที่เปลี่ยนไปคือสนามสอบที่ง่ายลง 50 เท่า (5,000 -> 100 ตัวเลือก)")

    # ---------------- ส่วนที่ 2
    print("\n" + "=" * 78)
    print("ส่วนที่ 2  ตรวจสอบสูตรอันดับคาดหวังกับการจำลอง (ตารางที่ 2.11)")
    print("=" * 78)
    rows = []
    for true_rank in [3, 12, 60, 200, 500]:
        for m in [99, 999]:
            theo = 1 + (true_rank - 1) * m / (N_ITEMS - 1)
            tr = np.array([sampled_rank(true_rank, N_ITEMS, m, rng) for _ in range(8000)])
            sc = np.array([ndcg_from_rank(int(r), K) for r in tr])
            rows.append({"อันดับจริง": true_rank, "m": m, "ทฤษฎี": theo,
                         "จำลอง": tr.mean(), "อยู่ใน Top-10 (%)": (tr <= K).mean() * 100,
                         "sampled nDCG@10": sc.mean(),
                         "full nDCG@10": ndcg_from_rank(true_rank, K)})
    tab = pd.DataFrame(rows)
    display(tab.round(4))
    print("  ทฤษฎีตรงกับการจำลองทุกกรณี -> ใช้สูตรประมาณผลกระทบล่วงหน้าได้โดยไม่ต้องรันจริง")
    print("  สังเกตแถว 'อันดับจริง = 12' ที่หลุด Top-10 จริง แต่ได้ sampled nDCG@10 เกือบสมบูรณ์แบบ")

    # ---------------- ส่วนที่ 3  Winner Flip
    print("\n" + "=" * 78)
    print("ส่วนที่ 3  การสลับผู้ชนะ (Winner Flip): Model P เทียบ Model Q")
    print("=" * 78)
    N_ITEMS3, N_USERS = 500, 800
    print(f"\n  คลัง {N_ITEMS3} รายการ ผู้ใช้ {N_USERS} คน K = {K}")
    print("     Model P : อันดับจริง = 12 สำหรับผู้ใช้ทุกคน  (สม่ำเสมอ แต่หลุด Top-10 เสมอ)")
    print("     Model Q : อันดับ 3 สำหรับผู้ใช้ 30% และอันดับ 60 สำหรับ 70%")
    print("               (เก่งมากกับบางกลุ่ม แย่มากกับที่เหลือ)")

    ranks_P = np.full(N_USERS, 12)
    n30 = int(0.30 * N_USERS)
    ranks_Q = np.concatenate([np.full(n30, 3), np.full(N_USERS - n30, 60)])

    print(f"\n{'m':>16}{'nDCG@10 ของ P':>18}{'nDCG@10 ของ Q':>18}{'ผู้ชนะ':>10}{'อัตราส่วน':>14}")
    print("-" * 78)
    results, ms, pv, qv = {}, [], [], []
    for m in [20, 50, 100, 200, N_ITEMS3 - 1]:
        if m == N_ITEMS3 - 1:
            sP = np.array([ndcg_from_rank(int(r), K) for r in ranks_P])
            sQ = np.array([ndcg_from_rank(int(r), K) for r in ranks_Q])
            tag = f"{m} (เต็มคลัง)"
        else:
            sP = np.array([ndcg_from_rank(sampled_rank(int(r), N_ITEMS3, m, rng), K) for r in ranks_P])
            sQ = np.array([ndcg_from_rank(sampled_rank(int(r), N_ITEMS3, m, rng), K) for r in ranks_Q])
            tag = str(m)
        p, q = sP.mean(), sQ.mean()
        win = "P" if p > q else ("Q" if q > p else "เสมอ")
        rstr = f"{p/q:.2f} เท่า" if q > 0 else "P/0"
        results[m] = (p, q, win)
        ms.append(m); pv.append(p); qv.append(q)
        print(f"{tag:>16}{p:>18.4f}{q:>18.4f}{win:>10}{rstr:>14}")

    pf, qf, _ = results[N_ITEMS3 - 1]
    p100, q100, _ = results[100]
    print("\n  ข้อสรุปที่ต้องอ่านช้า ๆ")
    print(f"     ที่ m = 100 (ค่าที่ใช้แพร่หลายที่สุดในวรรณกรรม)")
    print(f"        P = {p100:.4f} ชนะ Q = {q100:.4f} ด้วยอัตราส่วน {p100/q100:.2f} เท่า")
    print(f"     ที่จัดอันดับเต็มคลัง")
    print(f"        P = {pf:.4f} แพ้ Q = {qf:.4f} อย่างสิ้นเชิง")
    print("     -> ผู้ชนะสลับกัน และไม่ใช่ความบังเอิญเชิงสถิติ")
    print("        แต่เป็นผลเชิงโครงสร้างของ Hard Cutoff ที่ K")
    print("        อันดับ 12 ของ P ถูกการสุ่ม 'อุ้ม' เข้า Top-10 อย่างสม่ำเสมอ")
    print("        ขณะที่อันดับ 60 ของผู้ใช้ 70% ของ Q ไกลเกินกว่าจะถูกอุ้ม")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(range(1, 31), [ndcg_from_rank(r, K) for r in range(1, 31)],
               marker="o", ms=3, color="#4C78A8")
    ax[0].axvline(K + 0.5, color="crimson", ls="--", lw=1.2)
    ax[0].annotate("hard cutoff at K=10", xy=(10.5, 0.29), xytext=(14, 0.6),
                   arrowprops=dict(arrowstyle="->", color="crimson"), color="crimson", fontsize=8)
    ax[0].set_title("nDCG@10 is NOT linear in rank")
    ax[0].set_xlabel("true rank"); ax[0].set_ylabel("nDCG@10")
    x = np.arange(len(ms))
    ax[1].bar(x - 0.18, pv, 0.36, label="Model P", color="#4C78A8")
    ax[1].bar(x + 0.18, qv, 0.36, label="Model Q", color="#F58518")
    ax[1].set_xticks(x); ax[1].set_xticklabels([str(m) for m in ms[:-1]] + ["full"])
    ax[1].set_xlabel("m (number of sampled negatives)")
    ax[1].set_ylabel("nDCG@10"); ax[1].set_title("Winner flip")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 78)
    print("ข้อกำหนดห้าข้อเมื่อจำเป็นต้องใช้การประเมินแบบสุ่ม (หัวข้อ 2.9.5)")
    print("=" * 78)
    for i, s in enumerate([
        "ระบุจำนวนรายการเชิงลบ m และวิธีสุ่ม (สม่ำเสมอ / ตามความนิยม / ในหมวดเดียวกัน)",
        "เรียกค่าที่ได้ว่า sampled nDCG@K ไม่ใช่ nDCG@K เฉย ๆ",
        "ถ้าทรัพยากรเอื้ออำนวย ให้รายงานทั้งสองค่า และตรวจว่าอันดับของแบบจำลองตรงกันหรือไม่",
        "ห้ามเทียบค่าที่ m ต่างกัน และห้ามเทียบ sampled metric ของตนกับ full metric ของงานอื่น",
        "ถ้า m < 50 ให้ถือว่าเป็นการคัดกรองเบื้องต้น ไม่ใช่ข้อสรุปที่ใช้ตัดสินได้",
    ], 1):
        print(f"  {i}. {s}")


exp_2_4()

In [ ]:
# ---------------------------------------------------------------------------
# ทดลองเอง 3 : เปลี่ยนอันดับจริงและจำนวนรายการเชิงลบ แล้วดูว่าคะแนนกระโดดแค่ไหน
# ---------------------------------------------------------------------------
TRUE_RANK, N_ITEMS, K = 500, 5000, 10        # <-- ลองแก้สามค่านี้

rng = np.random.default_rng(0)
print(f"จัดอันดับเต็มคลัง : nDCG@{K} = {ndcg_from_rank(TRUE_RANK, K):.4f}\n")
print(f"{'m':>8}{'E[rank] ทฤษฎี':>18}{'E[rank] จำลอง':>18}{'sampled nDCG@10':>20}")
print("-" * 64)
for m in [19, 49, 99, 199, 499, 999]:
    theo = 1 + (TRUE_RANK - 1) * m / (N_ITEMS - 1)
    tr = np.array([sampled_rank(TRUE_RANK, N_ITEMS, m, rng) for _ in range(4000)])
    sc = np.mean([ndcg_from_rank(int(r), K) for r in tr])
    print(f"{m:>8}{theo:>18.2f}{tr.mean():>18.2f}{sc:>20.4f}")
print("\nยิ่ง m เล็ก สนามสอบยิ่งง่าย คะแนนยิ่งเฟ้อ — และไม่หายไปจนกว่า m = ขนาดคลัง")

---
# ส่วนที่ 5 — คุณภาพนอกเหนือความถูกต้อง (Beyond-Accuracy)
### (เทียบเท่า `lab2_05_beyond_accuracy.py` · ตำราหัวข้อ 2.5)

**คำถามวิจัย** ระบบที่แม่นยำที่สุด เป็นระบบที่ดีที่สุดสำหรับผู้ใช้และธุรกิจจริงหรือไม่

ส่วนนี้แบ่งเป็นสองระดับ

**5.1 MovieMini-12 — คำนวณด้วยมือได้ทุกตัวเลข** (ตัวอย่าง 2.6 ถึง 2.12 ในตำรา)
คลัง 12 เรื่อง ผู้ใช้ 8 คน K = 5 ช่องแสดงผลทั้งระบบ 8 × 5 = 40 ช่อง

| | Model X (Popularity) | Model Y (Personalized) |
|---|---|---|
| แนวคิด | ยอดนิยมชุดเดียวให้ทุกคน | ปรับตามรสนิยมรายคน |

**5.2 RSLab-Eval — แบบจำลองสามตัวบนคลัง 500 รายการ**
`TopPopular` · `Personalized` · `Diversified` — ไม่มีตัวใดชนะทุกคอลัมน์

**สิ่งที่ต้องสังเกตที่สุด** — ผลลัพธ์ที่ขัดสัญชาตญาณสองข้อ

1. **Model X ได้ ILD สูงกว่า Model Y** ทั้งที่ยัดรายการชุดเดียวกันให้ทุกคน
   เพราะ ILD วัดความหลากหลาย *ภายในรายการของผู้ใช้คนเดียว* จึงมองไม่เห็นความซ้ำซากข้ามผู้ใช้
2. **Model X ได้ Parity Gap = 0.0000 "สมบูรณ์แบบ"** ทั้งที่แย่พอ ๆ กันกับทุกคน (*Leveling Down*)

> บทเรียน: หากเลือกรายงานเฉพาะ ILD กับ Parity Gap ผู้อ่านจะสรุปกลับด้านจากความจริง
> **การเลือกตัวชี้วัดมีผลต่อข้อสรุปพอ ๆ กับการเลือกแบบจำลอง**

In [ ]:
# ============================================================================
# 5.1  MovieMini-12 : คำนวณตัวชี้วัด Beyond-Accuracy ทีละขั้น  (lab2_05 ส่วนที่ 0)
# ============================================================================
MM_US = ["Emma", "James", "Olivia", "Michael", "Sophia", "Daniel", "Ava", "Ethan"]
MM_K = 5

# Model X : ยอดนิยมล้วน ทุกคนได้ชุดเดียวกันหมด
MM_TOP5 = ["i1", "i4", "i7", "i10", "i2"]
MM_X = {u: list(MM_TOP5) for u in MM_US}

# Model Y : เฉพาะบุคคล แต่ละคนได้คนละชุด
MM_Y = {
    "Emma":    ["i1", "i2", "i3", "i9", "i12"],
    "James":   ["i1", "i2", "i3", "i10", "i5"],
    "Olivia":  ["i4", "i5", "i6", "i11", "i9"],
    "Michael": ["i3", "i7", "i8", "i2", "i1"],
    "Sophia":  ["i4", "i5", "i6", "i10", "i12"],
    "Daniel":  ["i2", "i3", "i9", "i7", "i8"],
    "Ava":     ["i2", "i4", "i6", "i11", "i10"],
    "Ethan":   ["i1", "i5", "i7", "i3", "i11"],
}
# กลุ่มผู้ใช้ตามความยาวประวัติ (ให้คะแนนไว้ ≥ 8 เรื่อง = ประวัติยาว)
MM_GROUP = {"ประวัติยาว": ["Emma", "Olivia", "Sophia", "Daniel"],
            "ประวัติสั้น": ["James", "Michael", "Ava", "Ethan"]}


def moviemini_demo():
    print("=" * 92)
    print("5.1  MovieMini-12  (ตัวอย่าง 2.6 ถึง 2.12 ในตำรา)")
    print("=" * 92)

    # เฉลย: เรื่องที่ผู้ใช้ให้คะแนน >= 4 ถือว่าเกี่ยวข้อง
    REL = ratings_small[ratings_small.rating >= 4].groupby("user_name")["item_id"].apply(set).to_dict()
    PROF = {}
    for u in MM_US:
        c = Counter(MM_GENRE[i] for i in REL[u])
        tot = sum(c.values())
        PROF[u] = {k: v / tot for k, v in c.items()}

    print("\n  คลัง MovieMini-12")
    print(f"     {'รหัส':<6}{'ชื่อเรื่อง':<18}{'แนว':<12}{'ค่าย':<10}{'ผู้เคยชม':>9}{'p(i)':>9}{'-log2 p':>10}")
    for i in MM_IDS:
        p = MM_POP[i] / MM_PLATFORM_USERS
        print(f"     {i:<6}{MM_TITLE[i]:<18}{MM_GENRE[i]:<12}{MM_STUDIO[i]:<10}"
              f"{MM_POP[i]:>9}{p:>9.3f}{-np.log2(p):>10.4f}")

    head, tail = head_tail_split(MM_POP, head_share=0.33)
    tot = sum(MM_POP.values())
    print(f"\n  [ตย. 2.8] เส้นแบ่งหัว–หาง : ปฏิสัมพันธ์รวม {tot:,}  หนึ่งในสาม = {tot/3:.2f}")
    acc = 0
    for it, c in sorted(MM_POP.items(), key=lambda kv: -kv[1]):
        acc += c
        print(f"     {it:<5}{MM_TITLE[it]:<18} ผู้เคยชม {c:>4}  สะสม {acc:>5}  "
              f"สัดส่วนสะสม {acc/tot:.4f}   -> {'หัว' if it in head else 'หาง'}")
    print(f"  กลุ่มหัว {len(head)} เรื่อง ({len(head)/12*100:.1f}% ของคลัง ครอง "
          f"{sum(MM_POP[i] for i in head)/tot*100:.1f}% ของปฏิสัมพันธ์)   กลุ่มหาง {len(tail)} เรื่อง")

    vp, wp, pp, gp = gini_trace([MM_POP[i] for i in MM_IDS])
    print(f"\n  จีนีของความนิยมในข้อมูลนำเข้า = {int(pp.sum()):,} / (12 × {int(vp.sum()):,})"
          f" = {gp:.4f}   <- เกณฑ์กำกับที่ผลลัพธ์ไม่ควรสูงกว่านี้")

    base = novelty(MM_X["Emma"], MM_POP, MM_PLATFORM_USERS)
    hack = novelty(["i1", "i4", "i7", "i10", "i12"], MM_POP, MM_PLATFORM_USERS)
    print(f"\n  ช่องโหว่ของ Novelty : ลิสต์เดิมของ Model X ได้ {base:.4f} บิต")
    print(f"     เมื่อแทน Inception ด้วย Spirited Away (ผู้ชม 2 คน) ได้ {hack:.4f} บิต"
          f"  สูงขึ้น {(hack/base-1)*100:.1f}% โดยเปลี่ยนรายการเดียว")

    summary = {}
    expo_store = {}
    for name, S in (("X", MM_X), ("Y", MM_Y)):
        lists = [S[u] for u in MM_US]
        cnt = Counter()
        for l in lists:
            cnt.update(l)
        expo = [cnt.get(i, 0) for i in MM_IDS]
        expo_store[name] = expo

        print("\n" + "-" * 92)
        print(f"  Model {name}")
        print("-" * 92)
        print("     การเปิดเผย :", "  ".join(f"{i}={cnt.get(i,0)}" for i in MM_IDS),
              f"  รวม {sum(expo)} (ต้องเท่ากับ 8×5 = 40)")

        cov = catalog_coverage(lists, 12)
        never = [i for i in MM_IDS if cnt.get(i, 0) == 0]
        print(f"\n     [ตย. 2.6 ] Coverage@5 = {12-len(never)}/12 = {cov:.4f}"
              f"   (ไม่ปรากฏเลย: {', '.join(never) or '—'})")

        v, w, prod, g = gini_trace(expo)
        print(f"\n     [ตย. 2.7 ] xᵢ เรียงแล้ว : {[int(x) for x in v]}")
        print(f"                2i − n − 1   : {[int(x) for x in w]}   (Σ = {int(w.sum())} ✓)")
        print(f"                ผลคูณ         : {[int(x) for x in prod]}  ->  Σ = {int(prod.sum())}")
        print(f"                Gini = {int(prod.sum())} / (12 × {int(v.sum())}) = {g:.4f}"
              f"   [ตรวจกับฟังก์ชัน: {gini_index(lists, 12):.4f}]")

        lt = [long_tail_ratio(S[u], tail) for u in MM_US]
        print(f"\n     [ตย. 2.8 ] LongTail@5 ต่อคน : {[round(x,2) for x in lt]}  ->  {np.mean(lt):.4f}")

        sim = category_sim(MM_GENRE)
        ild = [intra_list_diversity(S[u], sim) for u in MM_US]
        print(f"     [ตย. 2.9 ] ILD@5 ต่อคน      : {[round(x,2) for x in ild]}  ->  {np.mean(ild):.4f}")

        nv = [novelty(S[u], MM_POP, MM_PLATFORM_USERS) for u in MM_US]
        print(f"     [ตย. 2.10] Novelty@5 ต่อคน  : {[round(x,2) for x in nv]}  ->  {np.mean(nv):.4f}")

        cl = [calibration_error(S[u], REL[u], MM_GENRE) for u in MM_US]
        print(f"     [ตย. 2.11] CalibErr ต่อคน   : {[round(x,2) for x in cl]}  ->  {np.mean(cl):.4f}")

        pk = {u: len(set(S[u]) & REL[u]) / MM_K for u in MM_US}
        gm = {g2: float(np.mean([pk[u] for u in us])) for g2, us in MM_GROUP.items()}
        gap = group_parity_gap({g2: [pk[u] for u in us] for g2, us in MM_GROUP.items()})
        print(f"\n     [ตย. 2.12] P@5 ต่อคน : " + "  ".join(f"{u}={pk[u]:.1f}" for u in MM_US)
              + f"  ->  {np.mean(list(pk.values())):.4f}")
        print("                ค่าเฉลี่ยรายกลุ่ม : "
              + "   ".join(f"{g2} {v2:.4f}" for g2, v2 in gm.items())
              + f"   ParityGap = {gap:.4f}")

        pv = Counter({p2: 0 for p2 in set(MM_STUDIO.values())})
        for l in lists:
            for i in l:
                pv[MM_STUDIO[i]] += 1
        v2, w2, pr2, g2v = gini_trace(list(pv.values()))
        print(f"                การเปิดเผยรายค่าย : {dict(sorted(pv.items()))}")
        print(f"                Provider Gini = {int(pr2.sum())} / ({len(v2)} × {int(v2.sum())})"
              f" = {g2v:.4f}   [ตรวจกับฟังก์ชัน: {provider_exposure_gap(lists, MM_STUDIO):.4f}]")
        nz = [c for c in pv.values() if c > 0]
        if len(nz) < len(pv):
            _, _, prz, gz = gini_trace(nz)
            print(f"                ** กับดัก: ถ้าตัดค่ายที่ได้ศูนย์ทิ้ง จะได้ {gz:.4f}"
                  f" ซึ่งต่ำกว่าความจริง {g2v/gz:.1f} เท่า")
            print(f"                   กฎ: เวกเตอร์ต้องมีทุกหน่วยที่ 'มีสิทธิ์' ถูกเปิดเผย (ไม่ถูกแนะนำ = 0)")

        summary[name] = dict(P=np.mean(list(pk.values())), Cov=cov, Gini=g, LT=np.mean(lt),
                             ILD=np.mean(ild), Nov=np.mean(nv), Cal=np.mean(cl),
                             Gap=gap, PG=g2v)

    print("\n" + "=" * 92)
    print("  ตารางสรุปของ 5.1 (ตารางที่ 2.4 ในตำรา)")
    print("=" * 92)
    labels = [("P", "Precision@5"), ("Cov", "Coverage@5"), ("Gini", "Gini ของการเปิดเผย"),
              ("LT", "LongTail@5"), ("ILD", "ILD@5"), ("Nov", "Novelty@5 (บิต)"),
              ("Cal", "Calibration Error"), ("Gap", "Parity Gap"), ("PG", "Provider Gini")]
    tab = pd.DataFrame({"Model X (Popularity)": [summary["X"][k] for k, _ in labels],
                        "Model Y (Personalized)": [summary["Y"][k] for k, _ in labels]},
                       index=[lab for _, lab in labels])
    display(tab.round(4))

    print("     บทเรียนที่ 1  Model X ยัดรายการชุดเดียวกันให้ทุกคน แต่กลับได้ ILD *สูงกว่า* Model Y")
    print("                   เพราะห้าอันดับความนิยมแรกบังเอิญคาบเกี่ยวหลายแนว")
    print("                   ILD มองไม่เห็นความซ้ำซากข้ามผู้ใช้ ต้องดู Coverage และ Gini ควบคู่เสมอ")
    print("     บทเรียนที่ 2  Model X ได้ ParityGap = 0.0000 ซึ่งดู 'เป็นธรรมสมบูรณ์'")
    print("                   แต่เป็นความเท่าเทียมที่เกิดจากการให้บริการแย่เท่ากันทุกคน (Leveling Down)")
    print("                   จึงต้องรายงานค่าเฉลี่ยรายกลุ่มควบคู่กับช่องว่างเสมอ")

    # ---- กราฟ: การเปิดเผย + Lorenz Curve
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
    x = np.arange(12)
    ax[0].bar(x - 0.18, expo_store["X"], 0.36, label="Model X", color="#E45756")
    ax[0].bar(x + 0.18, expo_store["Y"], 0.36, label="Model Y", color="#4C78A8")
    ax[0].set_xticks(x); ax[0].set_xticklabels(MM_IDS, fontsize=8)
    ax[0].set_ylabel("times shown in Top-5"); ax[0].set_title("Exposure per item (40 slots)")
    ax[0].legend()
    for nm, col in (("X", "#E45756"), ("Y", "#4C78A8")):
        lx, ly = lorenz_points(expo_store[nm])
        ax[1].plot(lx, ly, marker="o", ms=3, color=col, label=f"Model {nm}")
    ax[1].plot([0, 1], [0, 1], ls="--", color="grey", lw=1, label="perfect equality")
    ax[1].set_xlabel("cumulative share of items"); ax[1].set_ylabel("cumulative share of exposure")
    ax[1].set_title("Lorenz curve (area × 2 = Gini)"); ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    return summary


mm_summary = moviemini_demo()

In [ ]:
# ============================================================================
# 5.2  RSLab-Eval : แบบจำลองสามตัวที่มี "บุคลิก" ต่างกันโดยเจตนา  (lab2_05 ส่วนหลัก)
# ============================================================================
def build_three_rankers(train, meta, K=10):
    pop = Counter(train.item_id)
    pop_order = [it for it, _ in pop.most_common()]
    cat_of = dict(zip(meta.item_id, meta.category))
    uc = (train.merge(meta[["item_id", "category"]], on="item_id")
               .groupby(["user_id", "category"]).size().reset_index(name="n")
               .sort_values(["user_id", "n"], ascending=[True, False]))
    fav = uc.groupby("user_id").head(2).groupby("user_id")["category"].apply(set).to_dict()

    def pop_ranker(u, cands):
        cs = set(cands)
        head = [i for i in pop_order if i in cs]
        hs = set(head)
        return head + [i for i in cands if i not in hs]

    def personal_ranker(u, cands):
        """คะแนน = ความนิยม แต่ดันหมวดโปรดของผู้ใช้ขึ้นก่อน"""
        f = fav.get(u, set())
        return sorted(cands, key=lambda i: -(pop.get(i, 0) * (5.0 if cat_of.get(i) in f else 1.0)))

    def diverse_ranker(u, cands):
        """จัดอันดับซ้ำแบบละโมบ: หักคะแนนรายการที่ซ้ำหมวดกับสิ่งที่เลือกไปแล้ว"""
        f = fav.get(u, set())
        base = {i: (pop.get(i, 0) * (5.0 if cat_of.get(i) in f else 1.0)) for i in cands}
        mx = max(base.values()) or 1.0
        base = {i: v / mx for i, v in base.items()}
        chosen, used, pool = [], Counter(), list(cands)
        for _ in range(min(K, len(pool))):
            best, bs = None, -1e9
            for i in pool:
                s = base[i] - 0.35 * used[cat_of.get(i)]
                if s > bs:
                    best, bs = i, s
            chosen.append(best); used[cat_of.get(best)] += 1; pool.remove(best)
        return chosen + pool

    return {"TopPopular": pop_ranker, "Personalized": personal_ranker,
            "Diversified": diverse_ranker}


def exp_2_5():
    K = 10
    df, meta = interactions, item_meta
    all_items = ALL_ITEMS
    train, _, test = global_temporal_split(df, train_frac=0.8)
    seen = train.groupby("user_id")["item_id"].apply(set).to_dict()
    rel = test[test.rating >= 4].groupby("user_id")["item_id"].apply(set).to_dict()
    pop_train = Counter(train.item_id)
    n_users_train = train.user_id.nunique()
    head, tail = head_tail_split(dict(pop_train), head_share=0.33)
    profile = train.groupby("user_id")["item_id"].apply(list).to_dict()
    cnt_u = train.groupby("user_id").size()
    med = cnt_u.median()
    grp = {u: ("ผู้ใช้เบา" if c <= med else "ผู้ใช้หนัก") for u, c in cnt_u.items()}

    print("=" * 92)
    print("5.2  ความถูกต้อง เทียบ คุณภาพนอกเหนือความถูกต้อง บน RSLab-Eval")
    print("=" * 92)
    print(f"\nคลัง {len(all_items)} รายการ | กลุ่มหัว {len(head)} รายการ "
          f"({len(head)/len(all_items)*100:.1f}%) | กลุ่มหาง {len(tail)} รายการ")
    print(f"ผู้ใช้ที่ประเมินได้ {len(rel):,} คน | K = {K} | Candidate Set = full catalog")

    sim_fn = category_sim(CAT_OF)
    rows = []
    for name, fn in build_three_rankers(train, meta, K).items():
        nd, reclists, ild, nov, ltr, cal = [], [], [], [], [], []
        by_group = {"ผู้ใช้เบา": [], "ผู้ใช้หนัก": []}
        for u, items in rel.items():
            cands = [i for i in all_items if i not in seen.get(u, set())]
            if not items & set(cands):
                continue
            top = fn(u, cands)[:K]
            score = ndcg_at_k(top, {i: 1.0 for i in items}, K)
            nd.append(score); reclists.append(top)
            ild.append(intra_list_diversity(top, sim_fn))
            nov.append(novelty(top, pop_train, n_users_train))
            ltr.append(long_tail_ratio(top, tail))
            cal.append(calibration_error(top, profile.get(u, []), CAT_OF))
            by_group[grp.get(u, "ผู้ใช้เบา")].append(score)
        rows.append({"Model": name, "nDCG@10": np.mean(nd),
                     "Coverage@10": catalog_coverage(reclists, len(all_items)),
                     "Gini": gini_index(reclists, len(all_items)),
                     "ILD": np.mean(ild), "Novelty": np.mean(nov),
                     "LongTail": np.mean(ltr), "CalibErr": np.mean(cal),
                     "ParityGap": group_parity_gap(by_group),
                     "ProviderGini": provider_exposure_gap(reclists, PROV_OF)})

    res = pd.DataFrame(rows).set_index("Model")
    print("\n" + "=" * 92)
    print("ตารางเปรียบเทียบ: ตัวชี้วัดหลักหนึ่งตัว และตัวชี้วัดกำกับแปดตัว")
    print("=" * 92)
    display(res.round(4))

    a, b, c = res.loc["TopPopular"], res.loc["Personalized"], res.loc["Diversified"]
    print("การอ่านตารางอย่างถูกวิธี\n")
    print(f"  TopPopular   nDCG@10 = {a['nDCG@10']:.4f} แต่ครอบคลุมคลังเพียง {a['Coverage@10']*100:.1f}%")
    print(f"               จีนีของผลลัพธ์สูงถึง {a['Gini']:.4f} -> คลังส่วนใหญ่ไม่มีโอกาสถูกค้นพบ")
    print(f"  Personalized nDCG@10 = {b['nDCG@10']:.4f} ({(b['nDCG@10']/a['nDCG@10']-1)*100:+.1f}%)"
          f" และ Coverage {b['Coverage@10']*100:.1f}% ({(b['Coverage@10']/a['Coverage@10']-1)*100:+.1f}%)")
    print(f"               -> เมื่อยังห่างจาก Pareto Frontier การปรับปรุงมักได้ทั้งสองด้าน")
    print(f"  Diversified  nDCG@10 = {c['nDCG@10']:.4f} ({(c['nDCG@10']/b['nDCG@10']-1)*100:+.1f}%"
          f" เทียบ Personalized)")
    print(f"               แลกมาด้วย ILD {b['ILD']:.4f} -> {c['ILD']:.4f}"
          f" และ LongTail {b['LongTail']:.4f} -> {c['LongTail']:.4f}")
    print("\n  จุดบอดของแต่ละตัวชี้วัดที่ต้องจำ")
    print("     Coverage ชนเพดานแล้วหยุดแยกความต่าง | Gini มองไม่เห็นว่าใครได้อะไร")
    print("     ILD มองไม่เห็นความซ้ำซากข้ามผู้ใช้   | Novelty ถูกปั่นได้ด้วยรายการเดียว")
    print("     Parity Gap ให้รางวัลเต็มแก่ระบบที่แย่กับทุกคนเท่ากัน")

    # ---- กราฟเรดาร์เชิงเปรียบเทียบ (normalize 0-1 ต่อคอลัมน์)
    show = ["nDCG@10", "Coverage@10", "ILD", "Novelty", "LongTail"]
    norm = res[show] / res[show].max()
    ang = np.linspace(0, 2 * np.pi, len(show), endpoint=False).tolist()
    ang += ang[:1]
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9),
                           gridspec_kw={"width_ratios": [1.2, 1]})
    res[show].plot(kind="bar", ax=ax[0], rot=0,
                   color=["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2"])
    ax[0].set_title("Raw values (different scales)")
    ax[0].legend(fontsize=7, ncol=2)
    ax[1] = fig.add_subplot(1, 2, 2, polar=True)
    for (idx, row), col in zip(norm.iterrows(), ["#4C78A8", "#F58518", "#54A24B"]):
        v = row.tolist(); v += v[:1]
        ax[1].plot(ang, v, color=col, label=idx, lw=1.6)
        ax[1].fill(ang, v, color=col, alpha=0.08)
    ax[1].set_xticks(ang[:-1]); ax[1].set_xticklabels(show, fontsize=7)
    ax[1].set_yticklabels([])
    ax[1].set_title("Normalised profile (1.0 = best in column)", fontsize=9)
    ax[1].legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=7)
    plt.tight_layout()
    plt.show()
    return res


res_2_5 = exp_2_5()

---
# ส่วนที่ 6 — ความไม่แน่นอนและการทดสอบนัยสำคัญ
### (เทียบเท่า `lab2_06_significance.py` · ตำราหัวข้อ 2.10)

**คำถามวิจัย** ส่วนต่างที่เห็นในตารางผล เป็นความจริงหรือเป็นเพียงสัญญาณรบกวน

**สี่การทดลอง**

1. **ความผันผวนจากเมล็ดสุ่มเพียงอย่างเดียว** — ฝึกแบบจำลองเดิมซ้ำ 8 ครั้ง เปลี่ยนแค่ seed
   > กฎที่ใช้ได้เสมอ: ถ้าส่วนต่างระหว่างสองการตั้งค่า **เล็กกว่า SD ระหว่าง seed ของการตั้งค่าเดียวกัน**
   > แสดงว่าไม่มีหลักฐานว่าต่างกัน
2. **Paired Bootstrap ระดับผู้ใช้** — สุ่มผู้ใช้กลับคืน 4,000 รอบ อ่านเปอร์เซ็นไทล์ 2.5 และ 97.5
3. **ความผิดพลาดคลาสสิก** — สุ่มซ้ำที่ระดับ *เหตุการณ์* แทนระดับ *ผู้ใช้* → ช่วงเชื่อมั่นแคบเกินจริง
4. **การเปรียบเทียบหลายคู่** — วิธีของ Holm–Bonferroni

**ประโยคที่เขียนได้ / เขียนไม่ได้**

| ✓ เขียนได้ | ✗ เขียนไม่ได้ |
|---|---|
| "ไม่มีหลักฐานว่า A ดีกว่า C บนชุดข้อมูลนี้" | "A ดีที่สุด" |
| "ทั้งสองดีกว่า B อย่างมีนัยสำคัญ" | "A ดีกว่า C เล็กน้อย" |

> ช่วงที่คร่อมศูนย์ **ไม่ได้แปลว่า A = C** — การไม่พบหลักฐานของความต่าง ไม่ใช่หลักฐานของความไม่ต่าง

In [ ]:
# ============================================================================
# การทดลองที่ 2.6  ความไม่แน่นอนและการทดสอบนัยสำคัญ  (lab2_06)
# ============================================================================
def exp_2_6():
    rng = np.random.default_rng(SEED)
    N_USERS = 600

    # ---------------- ส่วนที่ 1  ความผันผวนจากเมล็ดสุ่ม
    print("=" * 84)
    print("ส่วนที่ 1  ความผันผวนจากเมล็ดสุ่มเพียงอย่างเดียว (ตัวอย่าง 2.16)")
    print("=" * 84)
    print("\n  ฝึกแบบจำลองเดิมซ้ำ 8 ครั้ง เปลี่ยนเพียง Random Seed")
    print("  ไม่เปลี่ยนสถาปัตยกรรม ไฮเพอร์พารามิเตอร์ หรือข้อมูลใด ๆ ทั้งสิ้น")

    def train_once(seed):
        r = np.random.default_rng(1000 + seed)
        return float(np.clip(r.normal(0.0930, 0.0042), 0, 1))

    sv = seed_variance(train_once, seeds=range(8))
    print(f"\n  ค่าที่ได้ 8 ครั้ง        : {[round(v,4) for v in sv['values']]}")
    print(f"  ค่าเฉลี่ย               = {sv['mean']:.4f}")
    print(f"  ส่วนเบี่ยงเบนมาตรฐาน     = {sv['sd']:.4f}")
    print(f"  ช่วงที่พบ               = [{sv['range'][0]:.4f}, {sv['range'][1]:.4f}]  "
          f"กว้าง {sv['range'][1]-sv['range'][0]:.4f}")
    print(f"  ครึ่งความกว้าง CI 95%    = ±{sv['ci95_halfwidth']:.4f}")
    print(f"\n  งานตีพิมพ์จำนวนมากอ้างการปรับปรุง 1–3% ซึ่งบนฐาน {sv['mean']:.4f}")
    print(f"  คิดเป็นเพียง {sv['mean']*0.01:.4f}–{sv['mean']*0.03:.4f} เล็กกว่าความผันผวนจาก Seed เสียอีก")

    # ---------------- ส่วนที่ 2  Paired Bootstrap
    print("\n" + "=" * 84)
    print("ส่วนที่ 2  การสุ่มซ้ำแบบจับคู่ระดับผู้ใช้ (Paired Bootstrap)")
    print("=" * 84)
    base = rng.beta(2, 8, N_USERS)
    A = np.clip(base + rng.normal(0.020, 0.05, N_USERS), 0, 1)
    B = np.clip(base + rng.normal(0.000, 0.05, N_USERS), 0, 1)
    C = np.clip(base + rng.normal(0.018, 0.05, N_USERS), 0, 1)

    print(f"\n  nDCG@10 เฉลี่ย : A = {A.mean():.4f}   B = {B.mean():.4f}   C = {C.mean():.4f}")
    print("  ถ้าอ่านจากค่าเฉลี่ยอย่างเดียว จะสรุปว่า A > C > B")
    print("  แต่ค่าเฉลี่ยไม่ได้บอกว่าส่วนต่างนั้น 'ใหญ่พอที่จะเชื่อ' หรือไม่\n")

    print(f"{'คู่เปรียบเทียบ':<16}{'ส่วนต่างเฉลี่ย':>16}{'ช่วงเชื่อมั่น 95%':>28}"
          f"{'p-value':>12}{'ผู้ใช้ที่แย่ลง':>16}{'ข้อสรุป':>24}")
    print("-" * 116)
    pvals, boot_store = {}, {}
    for name, (x, y) in {"A − B": (A, B), "A − C": (A, C), "C − B": (C, B)}.items():
        r = paired_bootstrap(x, y, n_boot=4000, seed=7)
        pvals[name] = r["p_value"]
        boot_store[name] = r
        ci = f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"
        verdict = "มีนัยสำคัญ" if r["significant"] else "ไม่มีหลักฐานว่าต่างกัน"
        print(f"{name:<16}{r['mean_diff']:>+16.4f}{ci:>28}{r['p_value']:>12.4f}"
              f"{r['pct_users_worse']*100:>15.1f}%{verdict:>24}")

    print("\n  วิธีอ่าน: ช่วงเชื่อมั่นที่ 'คร่อมศูนย์' แปลว่าไม่มีหลักฐานว่าต่างกัน")
    print("  สิ่งที่ค่าเฉลี่ยปิดบัง: แม้ A ดีกว่า B โดยเฉลี่ย แต่ผู้ใช้ราว "
          f"{boot_store['A − B']['pct_users_worse']*100:.0f}% ได้ผล 'แย่ลง'")
    print("  การเปลี่ยนระบบอาจไม่คุ้มค่าแม้ค่าเฉลี่ยจะดีขึ้น")

    # ---------------- ส่วนที่ 3  สุ่มซ้ำผิดระดับ
    print("\n" + "=" * 84)
    print("ส่วนที่ 3  ความผิดพลาดคลาสสิก: สุ่มซ้ำผิดระดับ (ตัวอย่าง 2.17)")
    print("=" * 84)
    n_ev = 8
    ev_A = np.repeat(A, n_ev) + rng.normal(0, 0.01, N_USERS * n_ev)
    ev_B = np.repeat(B, n_ev) + rng.normal(0, 0.01, N_USERS * n_ev)
    r_user = paired_bootstrap(A, B, n_boot=4000, seed=7)
    r_event = paired_bootstrap(ev_A, ev_B, n_boot=4000, seed=7)
    w_user = r_user["ci_high"] - r_user["ci_low"]
    w_event = r_event["ci_high"] - r_event["ci_low"]
    print(f"\n  ✓ สุ่มซ้ำระดับผู้ใช้    (n = {r_user['n_users']:,})  ช่วงเชื่อมั่นกว้าง {w_user:.4f}")
    print(f"  ✗ สุ่มซ้ำระดับเหตุการณ์ (n = {r_event['n_users']:,})  ช่วงเชื่อมั่นกว้าง {w_event:.4f}")
    print(f"\n  ช่วงเชื่อมั่นแคบเกินจริง {w_user/w_event:.1f} เท่า = มั่นใจเกินความเป็นจริงเกือบสามเท่า")
    print("  เหตุผล: เหตุการณ์ของผู้ใช้คนเดียวกันไม่เป็นอิสระต่อกัน การสุ่มระดับเหตุการณ์")
    print("          จึงเท่ากับนับข้อมูลชุดเดิมซ้ำหลายครั้ง")
    print("  กฎ: หน่วยของการสุ่มซ้ำต้องเป็นหน่วยที่อิสระต่อกัน ซึ่งคือ 'ผู้ใช้' ไม่ใช่ 'เหตุการณ์'")

    # ---------------- ส่วนที่ 4  Holm
    print("\n" + "=" * 84)
    print("ส่วนที่ 4  การเปรียบเทียบหลายคู่และวิธีของ Holm–Bonferroni")
    print("=" * 84)
    print(f"\n  ที่ alpha = 0.05 และ 10 คู่ โอกาสพบ 'นัยสำคัญปลอม' อย่างน้อยหนึ่งคู่ = "
          f"{1-(1-0.05)**10:.1%}\n")
    corr = holm_correction(pvals, alpha=0.05)
    print(f"{'คู่':<12}{'p-value':>12}{'เกณฑ์ของ Holm':>20}{'ผลหลังแก้':>20}")
    print("-" * 64)
    for name, d in corr.items():
        print(f"{name:<12}{d['p']:>12.4f}{d['threshold']:>20.4f}"
              f"{('มีนัยสำคัญ' if d['significant'] else 'ไม่มีนัยสำคัญ'):>20}")
    print("\n  ยืนยันด้วย Wilcoxon Signed-Rank (ไม่ต้องสมมติการแจกแจง)")
    for name, (x, y) in {"A − B": (A, B), "A − C": (A, C)}.items():
        w = wilcoxon_signed_rank(x, y)
        print(f"     {name:<8} statistic = {w['statistic']:>12,.1f}   p = {w['p_value']:.4f}")
    print("  ตำราแนะนำ Paired Bootstrap เป็นหลัก ใช้ Wilcoxon ยืนยัน")
    print("  และรายงานขนาดผล (effect size) ควบคู่กับ p เสมอ")

    # ---------------- กราฟ
    d_ab = A - B
    rngb = np.random.default_rng(7)
    idx = rngb.integers(0, N_USERS, size=(4000, N_USERS))
    boots = d_ab[idx].mean(axis=1)
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
    ax[0].hist(d_ab, bins=50, color="#4C78A8")
    ax[0].axvline(0, color="crimson", lw=1.4)
    ax[0].set_title(f"Per-user difference A − B  ({(d_ab<0).mean()*100:.0f}% of users are worse off)")
    ax[0].set_xlabel("nDCG@10 difference")
    ax[1].hist(boots, bins=50, color="#54A24B")
    ax[1].axvline(0, color="crimson", lw=1.4)
    ax[1].axvline(np.percentile(boots, 2.5), color="black", ls="--", lw=1)
    ax[1].axvline(np.percentile(boots, 97.5), color="black", ls="--", lw=1)
    ax[1].set_title("Bootstrap distribution of the mean difference (4,000 resamples)")
    ax[1].set_xlabel("mean difference")
    plt.tight_layout()
    plt.show()


exp_2_6()

---
# ส่วนที่ 7 — โปรโตคอลการประเมิน 11 ขั้น
### (เทียบเท่า `lab2_07_full_protocol.py` · ตำราหัวข้อ 2.13–2.14)

รวมทุกอย่างในบทเข้าเป็นกระบวนงานเดียวที่ทำซ้ำได้ และปิดท้ายด้วย **บัตรโปรโตคอล**
ที่ต้องแนบไปกับตัวเลขทุกครั้งที่รายงานผล

| ขั้น | เนื้อหา | | ขั้น | เนื้อหา |
|---|---|---|---|---|
| 1 | กำหนดเป้าหมายผู้ใช้และธุรกิจ | | 7 | กำหนด Candidate Set |
| 2 | เลือก Primary Metric | | 8 | ระบุวิธี Negative Sampling |
| 3 | กำหนด Guardrail Metrics | | 9 | วัดทั้ง Accuracy และ Beyond-Accuracy |
| 4 | ระบุสูตรและค่า K | | 10 | รายงานความไม่แน่นอน |
| 5 | แบ่งข้อมูลตามเวลา | | 11 | ตรวจผลด้วยผู้ใช้จริง |
| 6 | ป้องกัน Data Leakage | | | |

> **ทุกขั้นต้องถูกกำหนดและบันทึกก่อนดูผลการทดลอง**
> สาเหตุที่ผลทำซ้ำไม่ได้ ไม่ใช่ความไม่ซื่อสัตย์ แต่คือตัวเลือกเชิงระเบียบวิธีจำนวนมาก
> ที่ไม่เคยถูกบันทึกในบทความ ทั้งที่แต่ละตัวเลือกเปลี่ยนผลได้อย่างมีนัยสำคัญ

In [ ]:
# ============================================================================
# การทดลองที่ 2.7  โปรโตคอลการประเมินครบ 11 ขั้น  (lab2_07)
# ============================================================================
def step(n, title):
    print(f"\n{'─'*80}\nขั้นที่ {n}  {title}\n{'─'*80}")


def exp_2_7():
    df, meta = interactions, item_meta
    all_items = ALL_ITEMS

    print("=" * 80)
    print("การทดลองที่ 2.7  โปรโตคอลการประเมินครบ 11 ขั้น")
    print("=" * 80)

    step(1, "กำหนดเป้าหมายของผู้ใช้และธุรกิจ")
    print("  เป้าหมายผู้ใช้ : พบรายการที่ตนชอบภายในสิบอันดับแรก โดยไม่ต้องเลื่อนหา")
    print("  เป้าหมายธุรกิจ : เพิ่มการค้นพบรายการใหม่ โดยไม่ลดความพึงพอใจระยะสั้น")
    print("  -> เป้าหมายทั้งสองข้อนี้เองที่กำหนดว่าตัวชี้วัดใดคือตัวหลัก")

    step(2, "เลือกตัวชี้วัดหลัก (Primary Metric)")
    print("  nDCG@10 เฉลี่ยแบบ macro-average ข้ามผู้ใช้ บน Full-Catalog Ranking")
    print("  โดยตัดรายการที่ผู้ใช้เคยมีปฏิสัมพันธ์ในชุดฝึกออกก่อน")
    print("  (การประกาศต้องครบทั้ง ชื่อ · ค่า K · วิธีเฉลี่ย · Candidate Set)")

    step(3, "กำหนดตัวชี้วัดกำกับ (Guardrail Metrics)")
    print("  Coverage@10 ≥ 0.30 | Gini ของผลลัพธ์ ≤ Gini ของข้อมูลนำเข้า")
    print("  Novelty ต้องไม่ลดลง | ParityGap ระหว่างกลุ่มผู้ใช้ ≤ 0.05")
    print("  เกณฑ์ต้องตรวจสอบด้วยโปรแกรมได้: 'ต้องไม่ซ้ำซากเกินไป' ใช้ไม่ได้")

    step(4, "ระบุสูตรและค่า K")
    K = 10
    print(f"  K = {K} | นิยามความเกี่ยวข้อง: rating >= 4 | gain_type = linear (rel ∈ {{0,1}})")

    step(5, "แบ่งข้อมูลตามเวลา")
    train, valid, test = global_temporal_split(df, train_frac=0.7, valid_frac=0.1)
    print(f"  ฝึก {len(train):,} | ตรวจสอบ {len(valid):,} | ทดสอบ {len(test):,}")

    step(6, "ป้องกันข้อมูลรั่วไหล")
    rep = leakage_report(train, test)
    print(f"  เหตุการณ์อนาคตในชุดฝึก = {rep['n_future_in_train']:,} ({rep['pct_future']:.1f}%)")
    print(f"  ผลตรวจ = {rep['verdict']}")
    print("  สถิติทุกตัว (ความนิยม ค่าเฉลี่ย พจนานุกรมรายการ) คำนวณจากชุดฝึกเท่านั้น")

    step(7, "กำหนดเซตผู้สมัคร (Candidate Set)")
    print("  candidate_set = full_catalog (ตัดรายการที่ผู้ใช้เห็นแล้วในชุดฝึกออก)")

    step(8, "ระบุวิธีสุ่มรายการเชิงลบ")
    print("  ไม่ใช้การสุ่ม จึงรายงานได้ว่าเป็น nDCG@10 เต็มรูปแบบ ไม่ต้องเติมคำว่า sampled")

    step(9, "วัดทั้งความถูกต้องและคุณภาพนอกเหนือความถูกต้อง")
    seen = train.groupby("user_id")["item_id"].apply(set).to_dict()
    rel = test[test.rating >= 4].groupby("user_id")["item_id"].apply(set).to_dict()
    pop = Counter(train.item_id)
    _, tail = head_tail_split(dict(pop), 0.33)
    rankers = build_three_rankers(train, meta, K)

    results, per_user = {}, {}
    for name in ["TopPopular", "Personalized"]:
        fn = rankers[name]
        nd, lists, nov, lt, users = [], [], [], [], []
        for u, items in rel.items():
            cands = [i for i in all_items if i not in seen.get(u, set())]
            if not items & set(cands):
                continue
            top = fn(u, cands)[:K]
            nd.append(ndcg_at_k(top, {i: 1.0 for i in items}, K))
            lists.append(top); users.append(u)
            nov.append(novelty(top, pop, train.user_id.nunique()))
            lt.append(long_tail_ratio(top, tail))
        per_user[name] = (users, np.array(nd))
        results[name] = {"nDCG@10": np.mean(nd),
                         "Coverage@10": catalog_coverage(lists, len(all_items)),
                         "Gini": gini_index(lists, len(all_items)),
                         "Novelty": np.mean(nov), "LongTail": np.mean(lt)}
    display(pd.DataFrame(results).T.round(4))

    step(10, "รายงานความไม่แน่นอน")
    ua, sa = per_user["Personalized"]
    ub, sb = per_user["TopPopular"]
    common = sorted(set(ua) & set(ub))
    ia = {u: i for i, u in enumerate(ua)}
    ib = {u: i for i, u in enumerate(ub)}
    xa = np.array([sa[ia[u]] for u in common])
    xb = np.array([sb[ib[u]] for u in common])
    r = paired_bootstrap(xa, xb, n_boot=4000, seed=11)
    print(f"  เปรียบเทียบบนผู้ใช้ร่วม {len(common):,} คน (สุ่มซ้ำระดับผู้ใช้ 4,000 รอบ)")
    print(f"  Personalized − TopPopular = {r['mean_diff']:+.4f}")
    print(f"  ช่วงความเชื่อมั่น 95% = [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]  p = {r['p_value']:.4f}")
    print(f"  ผู้ใช้ที่ได้ผลแย่ลง = {r['pct_users_worse']*100:.1f}%")
    print(f"  ข้อสรุป: {'มีนัยสำคัญ' if r['significant'] else 'ไม่มีหลักฐานว่าต่างกัน'}")

    step(11, "ตรวจผลด้วยผู้ใช้จริง")
    print("  ผลออฟไลน์ข้างต้นเป็นเพียง 'การคัดกรอง' เท่านั้น")
    print("  ก่อนนำขึ้นใช้จริงต้องผ่าน A/B Test พร้อมตัวชี้วัดกำกับเชิงธุรกิจ")
    print("  และต้องกำหนดเงื่อนไขการยกเลิกการทดลอง (rollback trigger) ไว้ล่วงหน้า")
    print("  กับดักที่พบบ่อย: Peeking · Novelty Effect · การละเลยผลระยะยาว")

    card = ProtocolCard(
        dataset="RSLab-Eval",
        n_users=int(df.user_id.nunique()), n_items=len(all_items),
        n_interactions=len(df), k_core=0,
        split_method="global_temporal", train_frac=0.7,
        relevance_threshold="rating >= 4", K=K,
        candidate_set="full_catalog", gain_type="linear",
        seeds=(0, 1, 2, 3, 4), primary_metric="nDCG@10",
        guardrail_metrics=("Coverage@10", "Gini", "Novelty", "LongTail"),
        notes="เปรียบเทียบ TopPopular กับ Personalized ด้วย paired bootstrap ระดับผู้ใช้")
    print("\n" + card.render())

    print("\n  ลองเปลี่ยนบัตรให้ใช้การสุ่มรายการเชิงลบ แล้วดูว่า validate() จับอะไรได้บ้าง")
    bad = ProtocolCard(split_method="random", candidate_set="sampled_negatives",
                       n_negatives=None, negative_sampling=None, seeds=(0,))
    for p in bad.validate():
        print(f"     ⚠ {p}")
    print(f"  ชื่อตัวชี้วัดที่ต้องรายงานเมื่อใช้การสุ่ม : "
          f"{ProtocolCard(candidate_set='sampled_negatives', n_negatives=99, negative_sampling='uniform').metric_name()}")


exp_2_7()

---
# ส่วนที่ 8 — เปรียบเทียบ Performance ระหว่างแบบจำลอง ⭐ (ส่วนที่เพิ่มใหม่)
### แบบจำลอง 5 ตัว · ตัวชี้วัด 11 ตัว · ทดสอบนัยสำคัญ · เวลาที่ใช้

ส่วนก่อนหน้าทั้งหมดสอนว่า *ตัวชี้วัดแต่ละตัวมองเห็นอะไร* ส่วนนี้นำทุกอย่างมาใช้จริง
กับคำถามที่ทีมพัฒนาต้องตอบทุกสัปดาห์ — **"แบบจำลองตัวไหนดีกว่ากัน และดีกว่าจริงหรือเปล่า"**

### แบบจำลองที่นำมาเปรียบเทียบ

| # | แบบจำลอง | หลักการ | บทบาทในการทดลอง |
|---|---|---|---|
| 1 | **Random** | สุ่มอันดับล้วน | เส้นฐานล่างสุด — ตัวชี้วัดใดที่แยก Random ออกจากตัวอื่นไม่ได้ ถือว่าใช้ไม่ได้ |
| 2 | **TopPopular** | ยอดนิยมชุดเดียวให้ทุกคน | เส้นฐานที่ "แข็งแรงอย่างน่าตกใจ" — เอาชนะยากกว่าที่คิดเสมอ |
| 3 | **ItemKNN** | ความคล้ายระหว่าง *รายการ* (โคไซน์) | Collaborative Filtering แบบเพื่อนบ้าน ฝั่งรายการ |
| 4 | **UserKNN** | ความคล้ายระหว่าง *ผู้ใช้* (โคไซน์, เพื่อนบ้าน 50 คน) | Collaborative Filtering แบบเพื่อนบ้าน ฝั่งผู้ใช้ |
| 5 | **MF-SVD** | แยกเมทริกซ์ด้วย Truncated SVD (32 ปัจจัยแฝง) | Model-based CF |

### โปรโตคอลที่ประกาศไว้ล่วงหน้า (ก่อนเห็นผล)

* **Primary Metric** — `nDCG@10` เฉลี่ยแบบ macro-average ข้ามผู้ใช้ บน **Full-Catalog Ranking**
* **Guardrail** — `Coverage@10 ≥ 0.30` · `Gini ≤ Gini ของข้อมูลนำเข้า` · `Novelty ต้องไม่ต่ำกว่า TopPopular`
* **การแบ่งข้อมูล** — Global Temporal Split 80/20 · **ความเกี่ยวข้อง** — `rating ≥ 4` ในชุดทดสอบ
* **Candidate Set** — คลังทั้งหมด ลบรายการที่ผู้ใช้เห็นแล้วในชุดฝึก (ไม่มีการสุ่มรายการเชิงลบ)
* **การทดสอบนัยสำคัญ** — Paired Bootstrap ระดับผู้ใช้ 4,000 รอบ + แก้หลายคู่ด้วย Holm

> ประกาศก่อน แล้วค่อยรัน — นี่คือความต่างระหว่างการประเมินกับการหาตัวเลขที่สวยที่สุดมารายงาน

In [ ]:
# ============================================================================
# 8.1  นิยามแบบจำลองทั้งห้าตัว
#
# ทุกตัวใช้อินเทอร์เฟซเดียวกัน เพื่อให้เปรียบเทียบได้อย่างเป็นธรรม
#     fit(R)                -> ฝึกจากเมทริกซ์ผู้ใช้ × รายการของ *ชุดฝึกเท่านั้น*
#     score_all()           -> คืนเมทริกซ์คะแนน (n_users × n_items)
# เมทริกซ์ R เป็นแบบโดยปริยาย (implicit): 1 = เคยมีปฏิสัมพันธ์, 0 = ไม่เคย
# ============================================================================
class BaseModel:
    name = "Base"

    def fit(self, R):
        raise NotImplementedError

    def score_all(self):
        raise NotImplementedError


class RandomModel(BaseModel):
    """เส้นฐานล่างสุด: สุ่มคะแนนให้ทุกคู่ (ผู้ใช้, รายการ)

    มีไว้เพื่อตอบคำถามว่า "ตัวชี้วัดนี้แยกของที่ไร้ความสามารถออกได้จริงไหม"
    ตัวชี้วัดใดที่ให้ Random ได้คะแนนใกล้เคียงตัวอื่น แปลว่าตัวชี้วัดนั้นไม่มีอำนาจจำแนก
    """
    name = "Random"

    def __init__(self, seed=0):
        self.seed = seed

    def fit(self, R):
        self.R = R
        return self

    def score_all(self):
        rng = np.random.default_rng(self.seed)
        return rng.random(self.R.shape)


class TopPopularModel(BaseModel):
    """ยอดนิยมชุดเดียวให้ทุกคน — ไม่มีสิ่งใดขึ้นกับตัวผู้ใช้เลย

    เส้นฐานที่แข็งแรงอย่างน่าตกใจในงานจริง เพราะความนิยมคือสัญญาณที่ทรงพลังที่สุด
    แต่จะแพ้ยับเยินในคอลัมน์ Coverage / Gini / Novelty
    """
    name = "TopPopular"

    def fit(self, R):
        self.pop = R.sum(axis=0)                       # นับจากชุดฝึกเท่านั้น
        self.n_users = R.shape[0]
        return self

    def score_all(self):
        return np.tile(self.pop, (self.n_users, 1))    # ทุกแถวเหมือนกันหมด


class ItemKNNModel(BaseModel):
    """ความคล้ายระหว่างรายการด้วยโคไซน์  score(u, i) = Σ_{j ∈ ประวัติของ u} sim(i, j)

    shrink ป้องกันไม่ให้รายการที่มีปฏิสัมพันธ์น้อยมากได้ค่าความคล้ายสูงเกินจริง
    ตัดความคล้ายของรายการกับตัวเองทิ้ง มิฉะนั้นจะแนะนำสิ่งที่ผู้ใช้เห็นแล้ว
    """
    name = "ItemKNN"

    def __init__(self, topk=100, shrink=5.0):
        self.topk, self.shrink = topk, shrink

    def fit(self, R):
        self.R = R
        norms = np.sqrt((R ** 2).sum(axis=0))
        num = R.T @ R                                   # (items × items)
        den = np.outer(norms, norms) + self.shrink + 1e-9
        S = num / den
        np.fill_diagonal(S, 0.0)
        if self.topk and self.topk < S.shape[1]:        # เก็บเพื่อนบ้านที่คล้ายที่สุด topk ตัว
            cut = np.partition(S, -self.topk, axis=1)[:, -self.topk][:, None]
            S = np.where(S >= cut, S, 0.0)
        self.S = S
        return self

    def score_all(self):
        return self.R @ self.S


class UserKNNModel(BaseModel):
    """ความคล้ายระหว่างผู้ใช้ด้วยโคไซน์  score(u, i) = Σ_{v ∈ เพื่อนบ้านของ u} sim(u, v)·R[v, i]"""
    name = "UserKNN"

    def __init__(self, topk=50, shrink=5.0):
        self.topk, self.shrink = topk, shrink

    def fit(self, R):
        self.R = R
        norms = np.sqrt((R ** 2).sum(axis=1))
        num = R @ R.T                                   # (users × users)
        den = np.outer(norms, norms) + self.shrink + 1e-9
        S = num / den
        np.fill_diagonal(S, 0.0)
        if self.topk and self.topk < S.shape[1]:
            cut = np.partition(S, -self.topk, axis=1)[:, -self.topk][:, None]
            S = np.where(S >= cut, S, 0.0)
        self.S = S
        return self

    def score_all(self):
        return self.S @ self.R


class MFSVDModel(BaseModel):
    """แยกเมทริกซ์ด้วย Truncated SVD:  R ≈ U_k Σ_k V_kᵀ

    ปัจจัยแฝง k ตัวบีบข้อมูลให้เหลือแก่นของรูปแบบการชอบ
    จุดแข็ง: จับความสัมพันธ์ที่เพื่อนบ้านโดยตรงมองไม่เห็น
    จุดอ่อน: k เป็นไฮเพอร์พารามิเตอร์ที่ต้องจูนบนชุดตรวจสอบ ไม่ใช่ชุดทดสอบ
    """
    name = "MF-SVD"

    def __init__(self, k=32):
        self.k = k

    def fit(self, R):
        self.mu = R.mean()
        Rc = R - self.mu
        U, s, Vt = np.linalg.svd(Rc, full_matrices=False)
        k = min(self.k, len(s))
        self.U, self.s, self.Vt = U[:, :k], s[:k], Vt[:k]
        return self

    def score_all(self):
        return (self.U * self.s) @ self.Vt + self.mu


MODEL_ZOO = [RandomModel(seed=SEED), TopPopularModel(),
             ItemKNNModel(topk=100), UserKNNModel(topk=50), MFSVDModel(k=32)]

print("นิยามแบบจำลองครบ 5 ตัว :", ", ".join(m.name for m in MODEL_ZOO))
print("\nทุกตัวเห็นข้อมูลชุดเดียวกัน แบ่งด้วยวิธีเดียวกัน และถูกวัดด้วยตัวชี้วัดชุดเดียวกัน")
print("ความต่างของตัวเลขที่จะเห็นในหัวข้อถัดไป จึงมาจาก 'ตัวแบบจำลอง' ล้วน ๆ")

In [ ]:
# ============================================================================
# 8.2  ตัวขับการประเมิน (Evaluation Harness) ที่ใช้ร่วมกันทุกแบบจำลอง
#
# ขั้นตอนตามโปรโตคอล 11 ขั้น
#   1) แบ่งข้อมูลตามเวลาก่อนคำนวณสถิติใด ๆ   2) สร้างเมทริกซ์จากชุดฝึกเท่านั้น
#   3) จัดอันดับเต็มคลัง ตัดรายการที่เห็นแล้ว  4) วัดทั้ง Accuracy และ Beyond-Accuracy
#   5) เก็บคะแนน "รายผู้ใช้" ไว้ทดสอบนัยสำคัญ
# ============================================================================
K_EVAL = 10
REL_TH = 4.0

# ---- ขั้นที่ 1  แบ่งข้อมูลตามเวลา (คำสั่งแรกเสมอ)
train_df, _, test_df = global_temporal_split(interactions, train_frac=0.8)

# ---- ขั้นที่ 2  สร้างเมทริกซ์และสถิติ *จากชุดฝึกเท่านั้น*
users = sorted(train_df.user_id.unique())
items = sorted(interactions.item_id.unique())
u_index = {u: i for i, u in enumerate(users)}
i_index = {it: j for j, it in enumerate(items)}
n_u, n_i = len(users), len(items)

R = np.zeros((n_u, n_i), dtype=float)
R[train_df.user_id.map(u_index).to_numpy(), train_df.item_id.map(i_index).to_numpy()] = 1.0

pop_train = Counter(train_df.item_id)
head_items, tail_items = head_tail_split(dict(pop_train), head_share=0.33)
sim_fn_cat = category_sim(CAT_OF)
gini_input = gini_trace([pop_train.get(it, 0) for it in items])[3]

# ---- เตรียม Ground Truth ของชุดทดสอบ
test_rel = (test_df[test_df.rating >= REL_TH]
            .groupby("user_id")["item_id"].apply(set).to_dict())
eval_users = [u for u in users if u in test_rel and len(test_rel[u]) > 0]

print(f"ผู้ใช้ในชุดฝึก           : {n_u:,}")
print(f"รายการในคลัง             : {n_i:,}")
print(f"ความหนาแน่นของเมทริกซ์    : {R.mean():.4f}")
print(f"ผู้ใช้ที่ประเมินได้        : {len(eval_users):,} คน (มีรายการ rating ≥ {REL_TH:.0f} ในชุดทดสอบ)")
print(f"กลุ่มหัว {len(head_items)} รายการ / กลุ่มหาง {len(tail_items)} รายการ")
print(f"Gini ของความนิยมในข้อมูลนำเข้า = {gini_input:.4f}   <- เกณฑ์กำกับของ Gini ผลลัพธ์")


def evaluate_model(model, K=K_EVAL, verbose=True):
    """ฝึก → ให้คะแนน → จัดอันดับเต็มคลัง → วัดผล  คืน (สรุป, คะแนนรายผู้ใช้, ลิสต์แนะนำ)"""
    t0 = time.perf_counter()
    model.fit(R)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    S = model.score_all()
    S = np.where(R > 0, -np.inf, S)          # ตัดรายการที่ผู้ใช้เห็นแล้วในชุดฝึกออก
    predict_time = time.perf_counter() - t0

    per_user, reclists, extra = {}, [], defaultdict(list)
    for u in eval_users:
        ui = u_index[u]
        s = S[ui]
        idx = np.argpartition(-s, K)[:K]
        idx = idx[np.argsort(-s[idx])]
        top = [items[j] for j in idx]
        relevant = test_rel[u]
        rel_map = {it: 1.0 for it in relevant}
        per_user[u] = {
            "nDCG@10": ndcg_at_k(top, rel_map, K),
            "MAP@10":  average_precision_at_k(top, relevant, K),
            "MRR@10":  reciprocal_rank(top, relevant, K),
            "P@10":    precision_at_k(top, relevant, K),
            "R@10":    recall_at_k(top, relevant, K),
            "HR@10":   hit_rate_at_k(top, relevant, K),
        }
        reclists.append(top)
        extra["ILD"].append(intra_list_diversity(top, sim_fn_cat))
        extra["Novelty"].append(novelty(top, pop_train, n_u))
        extra["LongTail"].append(long_tail_ratio(top, tail_items))

    keys = ["nDCG@10", "MAP@10", "MRR@10", "P@10", "R@10", "HR@10"]
    summary = {k: float(np.mean([per_user[u][k] for u in eval_users])) for k in keys}
    summary.update({
        "Coverage@10": catalog_coverage(reclists, n_i),
        "Gini": gini_index(reclists, n_i),
        "ILD": float(np.mean(extra["ILD"])),
        "Novelty": float(np.mean(extra["Novelty"])),
        "LongTail": float(np.mean(extra["LongTail"])),
        "fit_time_s": fit_time,
        "predict_time_s": predict_time,
    })
    if verbose:
        print(f"  {model.name:<12} ฝึก {fit_time:6.3f} วิ | ให้คะแนน {predict_time:6.3f} วิ"
              f" | nDCG@10 = {summary['nDCG@10']:.4f}")
    return summary, per_user, reclists


print("\nเริ่มประเมินแบบจำลองทั้งห้าตัว\n" + "-" * 70)
t_all = time.time()
SUMMARY, PER_USER, RECLISTS = {}, {}, {}
for m in MODEL_ZOO:
    s, pu, rl = evaluate_model(m)
    SUMMARY[m.name], PER_USER[m.name], RECLISTS[m.name] = s, pu, rl
print("-" * 70)
print(f"เสร็จทั้งหมดใน {time.time()-t_all:.1f} วินาที")

In [ ]:
# ============================================================================
# 8.3  ตารางเปรียบเทียบหลัก — ตัวชี้วัด 11 ตัว + เวลาที่ใช้
# ============================================================================
ORDER = ["Random", "TopPopular", "ItemKNN", "UserKNN", "MF-SVD"]
COLS_ACC = ["nDCG@10", "MAP@10", "MRR@10", "P@10", "R@10", "HR@10"]
COLS_BEY = ["Coverage@10", "Gini", "ILD", "Novelty", "LongTail"]
COLS_TIME = ["fit_time_s", "predict_time_s"]

board = pd.DataFrame(SUMMARY).T.loc[ORDER]

print("=" * 100)
print("ตารางที่ 8.1  ความถูกต้อง (Accuracy) — ตัวชี้วัดหลักคือ nDCG@10")
print("=" * 100)
display(board[COLS_ACC].round(4))

print("=" * 100)
print("ตารางที่ 8.2  คุณภาพนอกเหนือความถูกต้อง (Guardrail Metrics)")
print("=" * 100)
display(board[COLS_BEY].round(4))

print("=" * 100)
print("ตารางที่ 8.3  ต้นทุนเชิงระบบ (System Properties) — มักถูกละเลยแต่ตัดสินว่าขึ้นใช้งานจริงได้หรือไม่")
print("=" * 100)
display(board[COLS_TIME].round(4))

# ---------------------------------------------------------------- อ่านผล
primary = "nDCG@10"
winner = board[primary].idxmax()
base_pop = board.loc["TopPopular", primary]
base_rand = board.loc["Random", primary]

print("=" * 100)
print("การอ่านตารางตามโปรโตคอลที่ประกาศไว้")
print("=" * 100)
print(f"\n1) ผู้ชนะตาม Primary Metric ({primary}) คือ **{winner}** ที่ {board.loc[winner, primary]:.4f}")
print(f"   สูงกว่า TopPopular {(board.loc[winner, primary]/base_pop-1)*100:+.1f}%"
      f"  และสูงกว่า Random {board.loc[winner, primary]/max(base_rand,1e-9):.0f} เท่า")

print("\n2) ตรวจ Guardrail ทีละข้อ (เกณฑ์ประกาศไว้ก่อนเห็นผล)")
print(f"   {'แบบจำลอง':<14}{'Coverage@10 ≥ 0.30':>22}{'Gini ≤ ' + f'{gini_input:.4f}':>20}"
      f"{'Novelty ≥ TopPopular':>24}{'ผลรวม':>10}")
print("   " + "-" * 92)
nov_base = board.loc["TopPopular", "Novelty"]
verdicts = {}
for m in ORDER:
    c1 = board.loc[m, "Coverage@10"] >= 0.30
    c2 = board.loc[m, "Gini"] <= gini_input
    c3 = board.loc[m, "Novelty"] >= nov_base
    ok = c1 and c2 and c3
    verdicts[m] = ok
    print(f"   {m:<14}{board.loc[m,'Coverage@10']:>16.4f} {'✓' if c1 else '✗':>4}"
          f"{board.loc[m,'Gini']:>15.4f} {'✓' if c2 else '✗':>3}"
          f"{board.loc[m,'Novelty']:>19.4f} {'✓' if c3 else '✗':>3}"
          f"{'ผ่าน' if ok else 'ไม่ผ่าน':>10}")

print(f"\n   -> แม้ {winner} จะชนะ Primary Metric แต่การตัดสินใจขึ้นใช้งานจริง")
print(f"      ต้องดูว่าผ่าน Guardrail ครบทุกข้อด้วยหรือไม่: "
      f"{'ผ่าน' if verdicts[winner] else 'ไม่ผ่าน'}")
n_pass = sum(1 for m in ORDER if m != "Random" and verdicts[m])
if n_pass == 0:
    print("\n   ข้อสังเกตสำคัญ: ไม่มีแบบจำลองที่ 'ทำงานจริง' ตัวใดผ่านเกณฑ์ Gini เลย")
    print("   เพราะแบบจำลองที่เรียนรู้จากข้อมูลย่อมขยายอคติความนิยมที่มีอยู่แล้วในข้อมูล")
    print("   (Gini ของผลลัพธ์ > Gini ของข้อมูลนำเข้าเสมอ) มีเพียง Random เท่านั้นที่ผ่าน")
    print("   สิ่งที่ *ห้ามทำ* คือเห็นผลแล้วค่อยผ่อนเกณฑ์ให้ตัวที่ตนชอบผ่าน")
    print("   สิ่งที่ *ควรทำ* คือรายงานตามตรงว่าเกณฑ์นี้ไม่มีใครผ่าน แล้วอภิปรายว่า")
    print("   จะยอมรับระดับการกระจุกตัวเท่าใดในรอบถัดไป — และบันทึกการตัดสินใจนั้นไว้")

print("\n3) ข้อสังเกตที่ตารางเดียวบอกได้ทันที")
worst_cov = board["Coverage@10"].idxmin()
print(f"   * {worst_cov} ครอบคลุมคลังเพียง {board.loc[worst_cov,'Coverage@10']*100:.1f}%"
      f" และมี Gini = {board.loc[worst_cov,'Gini']:.4f}")
print(f"     -> คลังส่วนใหญ่ไม่เคยถูกแสดงเลยในการประเมินครั้งนี้")
print(f"   * Random ได้ Coverage สูงที่สุด ({board.loc['Random','Coverage@10']*100:.1f}%)"
      f" และ Gini ต่ำที่สุด ({board.loc['Random','Gini']:.4f})")
print("     -> เตือนว่า Coverage/Gini 'ดี' เพียงลำพังไม่ได้แปลว่าระบบดี")
print("        ต้องอ่านคู่กับ Primary Metric เสมอ ไม่เช่นนั้นระบบสุ่มจะชนะทุกครั้ง")
slow = board["fit_time_s"].idxmax()
print(f"   * แบบจำลองที่ฝึกนานที่สุดคือ {slow} ({board.loc[slow,'fit_time_s']:.3f} วินาที)"
      f" เทียบกับ TopPopular ({board.loc['TopPopular','fit_time_s']:.3f} วินาที)")
print("     -> บนคลัง 500 รายการยังไม่เป็นปัญหา แต่ ItemKNN/UserKNN ใช้หน่วยความจำ O(n²)")
print("        ซึ่งบนคลังหลักล้านรายการจะเป็นข้อจำกัดที่ตัดสินว่าขึ้น production ได้หรือไม่")

In [ ]:
# ============================================================================
# 8.4  กราฟเปรียบเทียบ — สี่มุมมองที่ควรอยู่ในรายงานทุกฉบับ
# ============================================================================
COLORS = {"Random": "#9E9E9E", "TopPopular": "#E45756", "ItemKNN": "#4C78A8",
          "UserKNN": "#54A24B", "MF-SVD": "#F58518"}
cols = [COLORS[m] for m in ORDER]

fig = plt.figure(figsize=(12.5, 8.2))

# (1) ตัวชี้วัดความถูกต้องทั้งหก
ax1 = fig.add_subplot(2, 2, 1)
x = np.arange(len(COLS_ACC))
w = 0.16
for i, m in enumerate(ORDER):
    ax1.bar(x + (i - 2) * w, board.loc[m, COLS_ACC].to_numpy(dtype=float), w,
            label=m, color=COLORS[m])
ax1.set_xticks(x)
ax1.set_xticklabels(COLS_ACC, fontsize=8)
ax1.set_title("(1) Accuracy metrics — the ranking of models is not identical across columns",
              fontsize=9)
ax1.legend(fontsize=7, ncol=2)

# (2) การแลกเปลี่ยนระหว่างความแม่นกับความครอบคลุม
ax2 = fig.add_subplot(2, 2, 2)
for m in ORDER:
    ax2.scatter(board.loc[m, "Coverage@10"], board.loc[m, "nDCG@10"],
                s=150, color=COLORS[m], edgecolor="white", zorder=3, label=m)
    ax2.annotate(m, (board.loc[m, "Coverage@10"], board.loc[m, "nDCG@10"]),
                 textcoords="offset points", xytext=(7, 5), fontsize=8)
ax2.axvline(0.30, ls="--", color="crimson", lw=1)
ax2.text(0.305, ax2.get_ylim()[0], " guardrail\n Coverage ≥ 0.30", color="crimson", fontsize=7)
ax2.set_xlabel("Coverage@10 (catalogue explored)")
ax2.set_ylabel("nDCG@10 (accuracy)")
ax2.set_title("(2) Accuracy vs catalogue coverage — no model wins both axes", fontsize=9)

# (3) โปรไฟล์ Beyond-Accuracy (ปรับให้เทียบกันได้)
ax3 = fig.add_subplot(2, 2, 3, polar=True)
radar = ["nDCG@10", "Coverage@10", "ILD", "Novelty", "LongTail"]
norm = board[radar] / board[radar].max()
ang = np.linspace(0, 2 * np.pi, len(radar), endpoint=False).tolist()
ang += ang[:1]
for m in ORDER:
    v = norm.loc[m].tolist()
    v += v[:1]
    ax3.plot(ang, v, color=COLORS[m], lw=1.6, label=m)
    ax3.fill(ang, v, color=COLORS[m], alpha=0.06)
ax3.set_xticks(ang[:-1])
ax3.set_xticklabels(radar, fontsize=7)
ax3.set_yticklabels([])
ax3.set_title("(3) Normalised profile (1.0 = best in column)", fontsize=9, pad=18)

# (4) ต้นทุนเวลา
ax4 = fig.add_subplot(2, 2, 4)
xt = np.arange(len(ORDER))
ax4.bar(xt - 0.2, board["fit_time_s"].to_numpy(dtype=float), 0.4, label="fit", color="#4C78A8")
ax4.bar(xt + 0.2, board["predict_time_s"].to_numpy(dtype=float), 0.4, label="score all users",
        color="#F58518")
ax4.set_xticks(xt)
ax4.set_xticklabels(ORDER, fontsize=8, rotation=12)
ax4.set_ylabel("seconds")
ax4.set_yscale("log")
ax4.set_title("(4) Training and scoring cost (log scale)", fontsize=9)
ax4.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- อันดับข้ามตัวชี้วัด
print("=" * 100)
print("อันดับของแบบจำลองในแต่ละตัวชี้วัด (1 = ดีที่สุด)")
print("=" * 100)
rank_tab = pd.DataFrame(index=ORDER)
for c in COLS_ACC + ["Coverage@10", "ILD", "Novelty", "LongTail"]:
    rank_tab[c] = board[c].rank(ascending=False).astype(int)
rank_tab["Gini"] = board["Gini"].rank(ascending=True).astype(int)      # ต่ำกว่าดีกว่า
display(rank_tab)

n_diff = rank_tab.apply(lambda col: col.idxmin()).nunique()
print(f"  มีแบบจำลองที่ 'ชนะอย่างน้อยหนึ่งคอลัมน์' ทั้งหมด {n_diff} ตัว จาก {len(ORDER)} ตัว")
print("  -> ยืนยันข้อสรุปของหัวข้อ 2.5 อีกครั้ง: ไม่มีแบบจำลองใดชนะทุกตัวชี้วัด")
print("     การเลือกจึงเป็น 'การตัดสินใจเชิงนโยบาย' ไม่ใช่การอ่านตัวเลขตัวเดียว")
print("     และนี่คือเหตุผลที่ต้องประกาศ Primary Metric ไว้ก่อนเห็นผล")

In [ ]:
# ============================================================================
# 8.5  ส่วนต่างที่เห็น เป็นของจริงหรือสัญญาณรบกวน
#      Paired Bootstrap ระดับผู้ใช้ + แก้การเปรียบเทียบหลายคู่ด้วย Holm
# ============================================================================
primary = "nDCG@10"
scores_by_model = {m: np.array([PER_USER[m][u][primary] for u in eval_users]) for m in ORDER}
champion = max(ORDER, key=lambda m: scores_by_model[m].mean())

print("=" * 104)
print(f"เปรียบเทียบทุกแบบจำลองกับผู้ชนะ ({champion}) ด้วย Paired Bootstrap 4,000 รอบ ระดับผู้ใช้")
print(f"ผู้ใช้ที่ใช้เปรียบเทียบ {len(eval_users):,} คน (คู่กันคนต่อคน)")
print("=" * 104)
print(f"\n{'คู่เปรียบเทียบ':<26}{'ส่วนต่างเฉลี่ย':>16}{'ช่วงเชื่อมั่น 95%':>28}"
      f"{'p-value':>11}{'ผู้ใช้ที่แย่ลง':>15}{'ข้อสรุป':>18}")
print("-" * 114)

pvals, boot_rows = {}, []
for m in ORDER:
    if m == champion:
        continue
    r = paired_bootstrap(scores_by_model[champion], scores_by_model[m], n_boot=4000, seed=11)
    key = f"{champion} − {m}"
    pvals[key] = r["p_value"]
    boot_rows.append({"pair": key, **r})
    ci = f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"
    verdict = "มีนัยสำคัญ" if r["significant"] else "ไม่มีหลักฐานว่าต่างกัน"
    print(f"{key:<26}{r['mean_diff']:>+16.4f}{ci:>28}{r['p_value']:>11.4f}"
          f"{r['pct_users_worse']*100:>14.1f}%{verdict:>18}")

print("\n" + "=" * 104)
print("แก้ปัญหาการเปรียบเทียบหลายคู่ด้วยวิธีของ Holm–Bonferroni (α = 0.05)")
print("=" * 104)
m_pairs = len(pvals)
print(f"  เปรียบเทียบ {m_pairs} คู่ -> โอกาสพบ 'นัยสำคัญปลอม' อย่างน้อยหนึ่งคู่ถ้าไม่แก้ = "
      f"{1-(1-0.05)**m_pairs:.1%}\n")
print(f"{'คู่':<26}{'p-value':>12}{'เกณฑ์ของ Holm':>18}{'ผลหลังแก้':>18}")
print("-" * 74)
for name, d in holm_correction(pvals, alpha=0.05).items():
    print(f"{name:<26}{d['p']:>12.4f}{d['threshold']:>18.4f}"
          f"{('มีนัยสำคัญ' if d['significant'] else 'ไม่มีนัยสำคัญ'):>18}")

print("\n  ยืนยันด้วย Wilcoxon Signed-Rank")
for m in ORDER:
    if m == champion:
        continue
    w = wilcoxon_signed_rank(scores_by_model[champion], scores_by_model[m])
    print(f"     {champion} − {m:<12} statistic = {w['statistic']:>12,.1f}   p = {w['p_value']:.4f}")

# ---- สิ่งที่ค่าเฉลี่ยปิดบัง
runner_up = sorted([m for m in ORDER if m != champion],
                   key=lambda m: -scores_by_model[m].mean())[0]
d = scores_by_model[champion] - scores_by_model[runner_up]
rngb = np.random.default_rng(11)
idx = rngb.integers(0, len(d), size=(4000, len(d)))
boots = d[idx].mean(axis=1)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].bar(ORDER, [scores_by_model[m].mean() for m in ORDER], color=cols)
for i, m in enumerate(ORDER):
    ax[0].text(i, scores_by_model[m].mean(), f"{scores_by_model[m].mean():.4f}",
               ha="center", va="bottom", fontsize=7)
ax[0].set_ylabel(primary)
ax[0].set_title(f"Mean {primary}", fontsize=9)
ax[0].tick_params(axis="x", rotation=15, labelsize=8)

ax[1].hist(d, bins=50, color="#4C78A8")
ax[1].axvline(0, color="crimson", lw=1.4)
ax[1].set_title(f"Per-user difference: {champion} − {runner_up}\n"
                f"{(d<0).mean()*100:.0f}% of users are worse off", fontsize=9)
ax[1].set_xlabel(primary + " difference")

ax[2].hist(boots, bins=50, color="#54A24B")
ax[2].axvline(0, color="crimson", lw=1.4)
ax[2].axvline(np.percentile(boots, 2.5), color="black", ls="--", lw=1)
ax[2].axvline(np.percentile(boots, 97.5), color="black", ls="--", lw=1)
ax[2].set_title("Bootstrap distribution of the mean difference\n(dashed = 95% CI)", fontsize=9)
plt.tight_layout()
plt.show()

print("\nสิ่งที่ค่าเฉลี่ยปิดบัง")
print(f"  แม้ {champion} ดีกว่า {runner_up} โดยเฉลี่ย แต่ผู้ใช้ {(d<0).mean()*100:.1f}% ได้ผลแย่ลง")
print("  การเปลี่ยนระบบอาจไม่คุ้มค่าแม้ค่าเฉลี่ยจะดีขึ้น -> ต้องดูการกระจายของผลต่าง ไม่ใช่ค่าเฉลี่ยอย่างเดียว")
print("\nประโยคที่เขียนได้ / เขียนไม่ได้")
print(f"  ✓ '{champion} ได้ {primary} สูงกว่า {runner_up} อย่างมีนัยสำคัญ (paired bootstrap ระดับผู้ใช้,")
print(f"     n = {len(eval_users):,}, Holm-corrected) บนชุดข้อมูล RSLab-Eval ภายใต้โปรโตคอลที่ประกาศไว้'")
print(f"  ✗ '{champion} เป็นแบบจำลองที่ดีที่สุด' — อ้างเกินกว่าที่ข้อมูลรองรับ")

In [ ]:
# ============================================================================
# 8.6  ผลของค่า K และผลของ "สนามสอบ" — สองปัจจัยที่เปลี่ยนผู้ชนะได้
# ============================================================================
Ks = [1, 3, 5, 10, 20, 50]
print("=" * 92)
print("8.6.1  nDCG@K ของแบบจำลองทั้งห้าเมื่อ K เปลี่ยน")
print("=" * 92)

sweep_rows = []
score_cache = {}
for mdl in MODEL_ZOO:
    mdl.fit(R)
    S = np.where(R > 0, -np.inf, mdl.score_all())
    score_cache[mdl.name] = S
    Kmax = max(Ks)
    for u in eval_users:
        ui = u_index[u]
        s = S[ui]
        idx = np.argpartition(-s, Kmax)[:Kmax]
        idx = idx[np.argsort(-s[idx])]
        top = [items[j] for j in idx]
        rel_map = {it: 1.0 for it in test_rel[u]}
        for k in Ks:
            sweep_rows.append({"Model": mdl.name, "K": k,
                               "nDCG@K": ndcg_at_k(top, rel_map, k),
                               "P@K": precision_at_k(top, test_rel[u], k),
                               "R@K": recall_at_k(top, test_rel[u], k)})

sweep = (pd.DataFrame(sweep_rows).groupby(["Model", "K"])[["nDCG@K", "P@K", "R@K"]]
         .mean().reset_index())
pivot = sweep.pivot(index="K", columns="Model", values="nDCG@K")[ORDER]
display(pivot.round(4))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for j, col in enumerate(["nDCG@K", "P@K", "R@K"]):
    p = sweep.pivot(index="K", columns="Model", values=col)[ORDER]
    for m in ORDER:
        ax[j].plot(p.index, p[m], marker="o", ms=4, color=COLORS[m], label=m)
    ax[j].set_xscale("log")
    ax[j].set_xticks(Ks)
    ax[j].set_xticklabels([str(k) for k in Ks])
    ax[j].set_xlabel("K")
    ax[j].set_title(col, fontsize=9)
ax[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

best_by_k = pivot.idxmax(axis=1)
print("  ผู้ชนะในแต่ละค่า K :")
for k in Ks:
    print(f"     K = {k:<3} -> {best_by_k.loc[k]}")
if best_by_k.nunique() > 1:
    print("  ** ผู้ชนะเปลี่ยนตามค่า K -> ยืนยันว่าต้องประกาศค่า K ไว้ล่วงหน้า และห้ามเทียบค่าที่ K ต่างกัน")
else:
    print("  ในชุดข้อมูลนี้ผู้ชนะไม่เปลี่ยนตาม K แต่ 'ระยะห่าง' ระหว่างแบบจำลองเปลี่ยนชัดเจน")
    print("  การเลือก K จึงยังกำหนดว่าเราจะสรุปว่า 'ดีกว่ากี่เท่า'")
print("  สังเกต Precision@K ลดลงเมื่อ K เพิ่ม ส่วน Recall@K ไม่มีวันลดลง — สมบัติเชิงโครงสร้าง")

# ------------------------------------------------------------------------
print("\n" + "=" * 92)
print("8.6.2  ถ้าเปลี่ยนไปใช้การสุ่มรายการเชิงลบ (Sampled Negatives) ผู้ชนะยังเหมือนเดิมหรือไม่")
print("=" * 92)
print("  ใช้แบบจำลองและข้อมูลชุดเดิมทุกประการ เปลี่ยนเพียง 'เซตผู้สมัคร' เท่านั้น\n")

rng_s = np.random.default_rng(SEED)
rows = []
for m_neg in [19, 49, 99, 499, None]:
    label = "full catalog" if m_neg is None else f"m = {m_neg}"
    for name in ORDER:
        S = score_cache[name]
        vals = []
        for u in eval_users:
            ui = u_index[u]
            rel = [it for it in test_rel[u] if i_index.get(it) is not None
                   and R[ui, i_index[it]] == 0]
            if not rel:
                continue
            if m_neg is None:
                cand = np.where(np.isfinite(S[ui]))[0]
            else:
                rel_idx = np.array([i_index[it] for it in rel])
                pool = np.where(np.isfinite(S[ui]))[0]
                pool = np.setdiff1d(pool, rel_idx, assume_unique=False)
                pick = rng_s.choice(pool, size=min(m_neg, len(pool)), replace=False)
                cand = np.concatenate([rel_idx, pick])
            sc = S[ui][cand]
            k = min(K_EVAL, len(cand))
            idx = cand[np.argsort(-sc)[:k]]
            top = [items[j] for j in idx]
            vals.append(ndcg_at_k(top, {it: 1.0 for it in rel}, K_EVAL))
        rows.append({"candidate_set": label, "Model": name, "nDCG@10": float(np.mean(vals))})

sampled = pd.DataFrame(rows).pivot(index="candidate_set", columns="Model",
                                   values="nDCG@10")[ORDER]
sampled = sampled.loc[["m = 19", "m = 49", "m = 99", "m = 499", "full catalog"]]
display(sampled.round(4))

winners = sampled.idxmax(axis=1)
print("  ผู้ชนะในแต่ละเซตผู้สมัคร :")
for lbl, w in winners.items():
    print(f"     {lbl:<14} -> {w}")
print(f"\n  คะแนนของ Random เฟ้อขึ้น "
      f"{sampled.loc['m = 19','Random']/max(sampled.loc['full catalog','Random'],1e-9):.0f} เท่า"
      " เมื่อสนามสอบเล็กลง ทั้งที่ไม่ได้ 'เก่งขึ้น' แม้แต่น้อย")
print("  (แถว m = 499 เท่ากับ full catalog เพราะเซตผู้สมัครที่เหลือหลังตัดรายการที่เห็นแล้ว")
print("   มีขนาดเล็กกว่า 499 อยู่แล้ว — การสุ่มจึงกินทั้งคลังพอดี)")
if winners.nunique() > 1:
    flips = [f"{l} -> {w}" for l, w in winners.items()]
    print(f"\n  ** ผู้ชนะสลับกัน (Winner Flip) เกิดขึ้นจริงในชุดข้อมูลนี้: {', '.join(flips)}")
    print("     ทีมที่รายงานผลที่ m = 19 กับทีมที่จัดอันดับเต็มบัญชี จะประกาศผู้ชนะคนละตัว")
    print("     ทั้งที่คำนวณถูกต้องทุกบรรทัด และใช้แบบจำลองชุดเดียวกันเป๊ะ")
print("\n  กฎที่ต้องจำ")
print("     * ถ้าใช้การสุ่ม ให้เรียกว่า sampled nDCG@10 (m=...) เสมอ")
print("     * ห้ามเทียบค่าที่ m ต่างกัน และห้ามเทียบ sampled metric กับ full metric ของงานอื่น")

In [ ]:
# ============================================================================
# 8.7  บัตรโปรโตคอลของการเปรียบเทียบครั้งนี้ + รายงานสรุปที่คัดลอกไปใช้ได้ทันที
# ============================================================================
card8 = ProtocolCard(
    dataset="RSLab-Eval (synthetic, seed=2569)",
    n_users=int(interactions.user_id.nunique()),
    n_items=n_i,
    n_interactions=len(interactions),
    k_core=0,
    split_method="global_temporal",
    train_frac=0.8,
    relevance_threshold="rating >= 4 (ในชุดทดสอบ)",
    K=K_EVAL,
    candidate_set="full_catalog",
    gain_type="linear",
    seeds=(2569,),
    primary_metric="nDCG@10",
    guardrail_metrics=("Coverage@10 >= 0.30", f"Gini <= {gini_input:.4f}",
                       "Novelty >= TopPopular"),
    notes=("เปรียบเทียบ Random / TopPopular / ItemKNN / UserKNN / MF-SVD "
           "ด้วย paired bootstrap ระดับผู้ใช้ 4,000 รอบ + Holm correction"))
print(card8.render())

print("\n" + "=" * 100)
print("รายงานสรุปของส่วนที่ 8 (คัดลอกไปวางในภาคผนวกได้)")
print("=" * 100)
best = board[primary].idxmax()
print(f"""
บนชุดข้อมูล RSLab-Eval (ผู้ใช้ {interactions.user_id.nunique():,} คน รายการ {n_i} ชิ้น
ปฏิสัมพันธ์ {len(interactions):,} ครั้ง) แบ่งข้อมูลแบบ Global Temporal Split ที่สัดส่วน 80/20
นิยามความเกี่ยวข้องจาก rating ≥ 4 และจัดอันดับเต็มบัญชีรายการโดยตัดรายการที่ผู้ใช้
เคยมีปฏิสัมพันธ์ในชุดฝึกออก ประเมินผู้ใช้ได้ {len(eval_users):,} คน

ตัวชี้วัดหลักที่ประกาศไว้ล่วงหน้าคือ nDCG@10 เฉลี่ยแบบ macro-average ข้ามผู้ใช้
ผลปรากฏว่า {best} มีค่าสูงสุดที่ {board.loc[best, primary]:.4f}
สูงกว่าเส้นฐาน TopPopular ({board.loc['TopPopular', primary]:.4f}) อยู่
{(board.loc[best, primary]/board.loc['TopPopular', primary]-1)*100:+.1f}%
และสูงกว่าเส้นฐาน Random ({board.loc['Random', primary]:.4f}) อย่างชัดเจน

การทดสอบนัยสำคัญด้วย paired bootstrap ระดับผู้ใช้ 4,000 รอบ พร้อมแก้การเปรียบเทียบ
หลายคู่ด้วยวิธี Holm–Bonferroni ที่ α = 0.05 ยืนยันผลข้างต้น โดยรายงานช่วงความเชื่อมั่น
ควบคู่กับสัดส่วนผู้ใช้ที่ได้ผลแย่ลง

ข้อจำกัดที่ต้องระบุ: ผลนี้ใช้ได้เฉพาะกับชุดข้อมูล ตัวชี้วัด ค่า K เซตผู้สมัคร และวิธีแบ่ง
ข้อมูลชุดนี้เท่านั้น หัวข้อ 8.6 แสดงว่าการเปลี่ยนค่า K หรือเปลี่ยนไปใช้การสุ่มรายการเชิงลบ
เปลี่ยนทั้งขนาดความต่างและอาจเปลี่ยนผู้ชนะได้ และคะแนนออฟไลน์ที่ดีขึ้นไม่รับประกัน
ความพึงพอใจของผู้ใช้จริง ต้องยืนยันด้วย A/B Test พร้อม Guardrail เชิงธุรกิจก่อนขึ้นใช้งาน
""")

---
# ส่วนที่ 9 — สรุปบทเรียนของทั้งเล่ม

### สิบข้อที่ต้องจำ

1. **MAE กับ RMSE ต้องรายงานคู่กัน** — ช่องว่างระหว่างสองค่าคือสัญญาณของความไม่สม่ำเสมอของความผิดพลาด
2. **ตัวชี้วัดที่ไม่ไวต่อตำแหน่งแยก A กับ B ไม่ออก** — P@K, R@K, F1@K, HR@K ให้ค่าเท่ากันเป๊ะ
3. **การเลือกตัวชี้วัดคือการเลือกข้อสรุป** — จึงต้องประกาศ Primary Metric ก่อนเห็นผล
4. **การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ** — ก่อนคำนวณสถิติใด ๆ แม้แต่ค่าเฉลี่ย
5. **การสุ่มรายการเชิงลบเปลี่ยนผู้ชนะได้** — ไม่ใช่แค่เปลี่ยนตัวเลข (Krichene & Rendle, 2020)
6. **ความแม่นยำเพียงด้านเดียวไม่พอ** — ต้องมี Guardrail ที่ตรวจด้วยโปรแกรมได้
7. **ตัวชี้วัดทุกตัวมีจุดบอด** — ILD มองไม่เห็นความซ้ำซากข้ามผู้ใช้ · Parity Gap ให้รางวัลแก่ Leveling Down
8. **ส่วนต่างที่เล็กกว่าความผันผวนจาก Seed ไม่ใช่การปรับปรุง**
9. **หน่วยของการสุ่มซ้ำคือ "ผู้ใช้" ไม่ใช่ "เหตุการณ์"**
10. **คะแนนออฟไลน์สูงไม่รับประกันความพึงพอใจ** — ต้องยืนยันด้วย A/B Test

---

### ขั้นต่อไป

| ต้องการ | ไปที่ |
|---|---|
| ฝึกเขียนเอง | `Lab2_Exercises.ipynb` — แบบฝึกหัด 12 ข้อ พร้อมเซลล์ตรวจคำตอบอัตโนมัติ |
| ดูเฉลยพร้อมคำอธิบาย | `Lab2_Exercises_Solutions.ipynb` |
| ทดลองต่อยอด | แก้ `MODEL_ZOO` ในส่วนที่ 8 แล้วเพิ่มแบบจำลองของตัวเองเข้าไปเปรียบเทียบ |

> **ประโยคปิดท้าย**
> คะแนนประเมินไม่ได้เป็นคุณสมบัติของแบบจำลองเพียงอย่างเดียว
> แต่เป็นผลร่วมของ **Model · ข้อมูล · ตัวชี้วัด · เวลา · Candidate Set · และโปรโตคอลการประเมินทั้งหมด**